# Ridge + LightGBM Rain-3 Rule With Optional Static Features

This notebook trains and predicts a full submission in one run.

What it does:
- Trains a joint 1D+2D ridge model for each `model_id`
- Trains rain-anchor LightGBM models for each `(model_id, node_type)` target
- Computes per-node closed-form blend weights from K-fold OOF predictions
- Builds the final submission and exports bundle artifacts in the same notebook

Static feature support:
- `ridge` can use static features through the same scalable projection/flat mechanism used in the joint static ridge notebook
- `LightGBM` can also use per-node raw static features as additional input columns
- Each path can be turned on or off independently

Important switches:
- `USE_STATIC_FEATURES_RIDGE=1/0`
- `USE_STATIC_FEATURES_LGBM=1/0`
- `STATIC_FEATURE_MODE=projection/flat/off`
- If both static switches are off, the notebook behaves like the original non-static rule-based version in terms of features

Outputs:
- `submission.csv` or target-suffixed submission file
- `oof_predictions`, `test_predictions`, `cv_summary`, and `bundle_manifest` when `SAVE_BUNDLE=1`


In [1]:
import os
import shutil
import time
import warnings
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor


warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but LGBMRegressor was fitted with feature names",
)

STD_DEV = {
    (1, 1): 16.877747,
    (1, 2): 14.378797,
    (2, 1): 3.191784,
    (2, 2): 2.727131,
}

KEY_COLS = ["model_id", "event_id", "node_type", "node_id", "timestep"]
SAFE_FEATURE_CLIP = 1e6
SAFE_LEVEL_CLIP = 1e6
SAFE_DELTA_CLIP = 1e3


def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, str(default)))


def env_float(name: str, default: float) -> float:
    return float(os.environ.get(name, str(default)))


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


def parse_targets(text: str):
    out = []
    for tok in text.split(","):
        tok = tok.strip()
        if not tok:
            continue
        parts = tok.split(":")
        if len(parts) != 2:
            raise ValueError(f"invalid TARGETS token: {tok}")
        out.append((int(parts[0]), int(parts[1])))
    if not out:
        raise ValueError("TARGETS is empty")
    return out


def parse_str_list(text: str):
    return [c.strip() for c in text.split(",") if c.strip()]


def target_suffix(targets) -> str:
    return "__".join(f"{int(model_id)}_{int(node_type)}" for model_id, node_type in targets)


def default_output_name(targets) -> str:
    suffix = target_suffix(targets)
    if suffix == "1_1__1_2__2_1__2_2":
        return "submission.csv"
    return f"submission__{suffix}.csv"


def default_bundle_name(targets) -> str:
    base = "ridge_lgbm_rain3_rule_static_optional"
    suffix = target_suffix(targets)
    if suffix == "1_1__1_2__2_1__2_2":
        return base
    return f"{base}__{suffix}"


def has_working_lgbm_gpu() -> bool:
    if shutil.which("nvidia-smi") is None:
        return False
    try:
        x_probe = np.array([[0.0], [1.0]], dtype=np.float32)
        y_probe = np.array([0.0, 1.0], dtype=np.float32)
        probe = LGBMRegressor(
            objective="regression",
            n_estimators=1,
            num_leaves=2,
            learning_rate=0.1,
            max_bin=63,
            device_type="gpu",
            verbosity=-1,
        )
        probe.fit(x_probe, y_probe)
        _ = probe.predict(x_probe)
        return True
    except Exception:
        return False


# =========================
# Config
# =========================
DATASET_ROOT = Path(os.environ.get("DATASET_ROOT", "/kaggle/input/datasets/mtmrs1/urban-flood-bench-re"))
EVENT_SPLIT_SEED = env_int("EVENT_SPLIT_SEED", 2026)
N_FOLDS = env_int("N_FOLDS", 5)
START_T = env_int("START_T", 10)
TARGETS = parse_targets(os.environ.get("TARGETS", "1:1,1:2"))
FORCE_METHOD = os.environ.get("FORCE_METHOD", "").strip().lower()
LAMBDA_CLIP_EPS = env_float("LAMBDA_CLIP_EPS", 1e-12)

RIDGE_ALPHA = env_float("RIDGE_ALPHA", 0.1)
NA_MODEL1 = env_int("NA_MODEL1", 3)
NA_MODEL2 = env_int("NA_MODEL2", 3)
NB_MODEL1 = env_int("NB_MODEL1", 100)
NB_MODEL2 = env_int("NB_MODEL2", 100)
USE_STATIC_FEATURES_RIDGE = env_bool("USE_STATIC_FEATURES_RIDGE", True)
USE_STATIC_FEATURES_LGBM = env_bool("USE_STATIC_FEATURES_LGBM", True)
STATIC_FEATURE_MODE = os.environ.get("STATIC_FEATURE_MODE", "projection").strip().lower()
STATIC_STANDARDIZE = env_bool("STATIC_STANDARDIZE", True)
STATIC_COLS_1D_REQ = parse_str_list(
    os.environ.get(
        "STATIC_COLS_1D",
        "position_x,position_y,depth,invert_elevation,surface_elevation,base_area",
    )
)
STATIC_COLS_2D_REQ = parse_str_list(
    os.environ.get(
        "STATIC_COLS_2D",
        "position_x,position_y,area,roughness,min_elevation,elevation,aspect,curvature,flow_accumulation",
    )
)

_device_env = os.environ.get("LGBM_DEVICE", "").strip().lower()
if _device_env in {"cpu", "gpu"}:
    LGBM_DEVICE = _device_env
    LGBM_DEVICE_SOURCE = "env"
else:
    LGBM_DEVICE = "gpu" if has_working_lgbm_gpu() else "cpu"
    LGBM_DEVICE_SOURCE = "auto"

Y_LAGS_1D = env_int("Y_LAGS_1D", 10)
Y_LAGS_2D = env_int("Y_LAGS_2D", 4)
RAIN_LAGS_MODEL1 = env_int("RAIN_LAGS_MODEL1", 60)
RAIN_LAGS_MODEL2 = env_int("RAIN_LAGS_MODEL2", 80)

LGBM_MAX_TRAIN_ROWS_1D = env_int("LGBM_MAX_TRAIN_ROWS_1D", env_int("MAX_TRAIN_ROWS_1D", 4_000_000))
LGBM_MAX_TRAIN_ROWS_2D = env_int("LGBM_MAX_TRAIN_ROWS_2D", env_int("MAX_TRAIN_ROWS_2D", 8_000_000))

N_ESTIMATORS_1D = env_int("N_ESTIMATORS_1D", 800)
N_ESTIMATORS_2D = env_int("N_ESTIMATORS_2D", 500)
LEARNING_RATE = env_float("LEARNING_RATE", 0.03)
NUM_LEAVES = env_int("NUM_LEAVES", 63)
MAX_DEPTH = env_int("MAX_DEPTH", -1)
MIN_CHILD_SAMPLES_1D = env_int("MIN_CHILD_SAMPLES_1D", 50)
MIN_CHILD_SAMPLES_2D = env_int("MIN_CHILD_SAMPLES_2D", 100)
MIN_DATA_IN_BIN = env_int("MIN_DATA_IN_BIN", 15)
SUBSAMPLE = env_float("SUBSAMPLE", 0.9)
COLSAMPLE_BYTREE = env_float("COLSAMPLE_BYTREE", 0.9)
REG_ALPHA = env_float("REG_ALPHA", 0.0)
REG_LAMBDA = env_float("REG_LAMBDA", 0.0)
LGBM_N_JOBS = env_int("LGBM_N_JOBS", 0)
MAX_BIN = env_int("MAX_BIN", 255)
MAX_CAT_THRESHOLD = env_int("MAX_CAT_THRESHOLD", 64)
GPU_USE_DP = env_bool("GPU_USE_DP", True)
NODE_POS_AS_CATEGORY = env_bool("NODE_POS_AS_CATEGORY", LGBM_DEVICE != "gpu")

RAIN_ANCHOR_QLOW = env_float("RAIN_ANCHOR_QLOW", 0.10)
RAIN_ANCHOR_QMID = env_float("RAIN_ANCHOR_QMID", 0.50)
RAIN_ANCHOR_QHIGH = env_float("RAIN_ANCHOR_QHIGH", 0.90)
RAIN_ANCHOR_BW_SCALE = env_float("RAIN_ANCHOR_BW_SCALE", 0.75)
RAIN_ANCHOR_MIN_BW = env_float("RAIN_ANCHOR_MIN_BW", 1e-4)

SAMPLE_SUBMISSION_PATH = os.environ.get("SAMPLE_SUBMISSION_PATH", "")
ALLOW_PARTIAL_TARGETS = env_bool("ALLOW_PARTIAL_TARGETS", True)
STRICT_PARTIAL_SUBMISSION = env_bool("STRICT_PARTIAL_SUBMISSION", False)
PARTIAL_SUBMISSION_FILL_VALUE = env_float("PARTIAL_SUBMISSION_FILL_VALUE", 0.0)
OUTPUT_PATH = Path(
    os.environ.get(
        "OUTPUT_PATH",
        str(Path("/kaggle/working") / default_output_name(TARGETS))
        if Path("/kaggle/working").exists()
        else default_output_name(TARGETS),
    )
)
SAVE_BUNDLE = env_bool("SAVE_BUNDLE", True)
EXPORT_OOF = env_bool("EXPORT_OOF", True)
FAIL_IF_OOF_EMPTY = env_bool("FAIL_IF_OOF_EMPTY", True)
SAVE_PARQUET = env_bool("SAVE_PARQUET", True)
OOF_MIN_MERGE_RATIO = env_float("OOF_MIN_MERGE_RATIO", 0.98)
OOF_LOG_EVERY_EVENTS = env_int("OOF_LOG_EVERY_EVENTS", 5)
DEFAULT_BUNDLE_ROOT = (
    "/kaggle/working/model_bundles" if Path("/kaggle/working").exists() else "ensemble/artifacts/model_bundles"
)
BUNDLE_ROOT = Path(os.environ.get("BUNDLE_ROOT", DEFAULT_BUNDLE_ROOT))
BUNDLE_NAME = os.environ.get("BUNDLE_NAME", default_bundle_name(TARGETS))
BUNDLE_DIR = Path(os.environ.get("BUNDLE_DIR", str(BUNDLE_ROOT / BUNDLE_NAME)))
DEFAULT_OOF_CACHE_DIR = OUTPUT_PATH.parent / f".oof_cache__{BUNDLE_NAME}"
OOF_CACHE_DIR = Path(os.environ.get("OOF_CACHE_DIR", str(DEFAULT_OOF_CACHE_DIR)))


if FORCE_METHOD not in {"", "ridge", "lgbm", "blend"}:
    raise ValueError(f"FORCE_METHOD must be one of '', 'ridge', 'lgbm', 'blend', got {FORCE_METHOD}")
if N_FOLDS < 2:
    raise ValueError(f"N_FOLDS must be >=2, got {N_FOLDS}")
if LAMBDA_CLIP_EPS <= 0.0:
    raise ValueError(f"LAMBDA_CLIP_EPS must be >0, got {LAMBDA_CLIP_EPS}")
if not (0.0 <= RAIN_ANCHOR_QLOW < RAIN_ANCHOR_QMID < RAIN_ANCHOR_QHIGH <= 1.0):
    raise ValueError(
        f"RAIN_ANCHOR quantiles must satisfy 0<=QLOW<QMID<QHIGH<=1, got "
        f"{RAIN_ANCHOR_QLOW}, {RAIN_ANCHOR_QMID}, {RAIN_ANCHOR_QHIGH}"
    )
if RAIN_ANCHOR_BW_SCALE <= 0.0:
    raise ValueError(f"RAIN_ANCHOR_BW_SCALE must be >0, got {RAIN_ANCHOR_BW_SCALE}")
if RAIN_ANCHOR_MIN_BW <= 0.0:
    raise ValueError(f"RAIN_ANCHOR_MIN_BW must be >0, got {RAIN_ANCHOR_MIN_BW}")
if NA_MODEL1 < 1 or NA_MODEL2 < 1:
    raise ValueError(f"NA_MODEL1 and NA_MODEL2 must be >=1, got {NA_MODEL1}, {NA_MODEL2}")
if NB_MODEL1 < 0 or NB_MODEL2 < 0:
    raise ValueError(f"NB_MODEL1 and NB_MODEL2 must be >=0, got {NB_MODEL1}, {NB_MODEL2}")
if STATIC_FEATURE_MODE not in {"projection", "flat", "off"}:
    raise ValueError(
        f"STATIC_FEATURE_MODE must be one of 'projection', 'flat', 'off', got {STATIC_FEATURE_MODE}"
    )


def resolve_models_root(dataset_root: Path) -> Path:
    def is_models_root(p: Path) -> bool:
        return (p / "Model_1" / "train").exists() and (p / "Model_2" / "train").exists()

    for p in (dataset_root / "Models", dataset_root):
        if is_models_root(p):
            return p

    kroot = Path("/kaggle/input")
    if kroot.exists():
        for p in kroot.rglob("Model_1"):
            cand = p.parent
            if (p / "train").exists() and (cand / "Model_2" / "train").exists():
                return cand

    raise RuntimeError(
        "Dataset path is invalid. Set DATASET_ROOT to the directory containing Models/Model_1 and Models/Model_2. "
        f"DATASET_ROOT={dataset_root}"
    )


def resolve_sample_submission(dataset_root: Path, models_root: Path, explicit: str):
    cands = []
    if explicit:
        cands.append(Path(explicit))
    cands.extend(
        [
            dataset_root / "sample_submission.csv",
            models_root / "sample_submission.csv",
            Path("/kaggle/input/competitions/urban-flood-modelling/sample_submission.csv"),
            Path("/kaggle/input/urban-flood-modelling/sample_submission.csv"),
            Path("/kaggle/input/flood-comp-dataset/sample_submission.csv"),
        ]
    )
    checked = []
    for p in cands:
        checked.append(str(p))
        if p.exists():
            return p
    checked_msg = ", ".join(checked[:16])
    raise RuntimeError(
        "sample submission not found. Set SAMPLE_SUBMISSION_PATH. "
        f"checked={checked_msg}"
    )


def model_dir(models_root: Path, model_id: int) -> Path:
    p = models_root / f"Model_{model_id}"
    if not (p / "train").exists() or not (p / "test").exists():
        raise RuntimeError(f"invalid model dir: {p}")
    return p


def node_dynamic_name(node_type: int) -> str:
    return "1d_nodes_dynamic_all.csv" if node_type == 1 else "2d_nodes_dynamic_all.csv"


def node_static_name(node_type: int) -> str:
    return "1d_nodes_static.csv" if node_type == 1 else "2d_nodes_static.csv"


def list_event_ids(mdir: Path, split: str):
    out = []
    for p in sorted((mdir / split).glob("event_*")):
        if not p.is_dir():
            continue
        try:
            out.append(int(p.name.split("_")[1]))
        except Exception:
            continue
    return out


def split_events(event_ids, ratio: float, seed: int):
    arr = np.asarray(event_ids, dtype=np.int32)
    rng = np.random.default_rng(seed)
    rng.shuffle(arr)
    n_train = int(len(arr) * ratio)
    n_train = max(1, min(n_train, len(arr) - 1))
    return sorted(arr[:n_train].tolist()), sorted(arr[n_train:].tolist())


def split_events_kfold(event_ids, n_folds: int, seed: int):
    arr = np.asarray(event_ids, dtype=np.int32)
    if arr.size < 2:
        raise RuntimeError("need at least 2 events for k-fold")
    rng = np.random.default_rng(seed)
    rng.shuffle(arr)
    k = max(2, min(int(n_folds), int(arr.size)))
    folds = []
    for a in np.array_split(arr, k):
        if a.size == 0:
            continue
        folds.append(sorted(a.astype(np.int32).tolist()))
    if len(folds) < 2:
        raise RuntimeError("k-fold split produced <2 folds")
    return folds


def fold_train_events(folds, holdout_idx: int):
    out = []
    for i, ids in enumerate(folds):
        if i == holdout_idx:
            continue
        out.extend(ids)
    return sorted(out)


def load_node_ids(mdir: Path, node_type: int):
    df = pd.read_csv(mdir / "train" / node_static_name(node_type), usecols=["node_idx"])
    return np.sort(df["node_idx"].astype(np.int32).unique())


def detect_rain_col(mdir: Path, split: str, event_id: int):
    cols = pd.read_csv(mdir / split / f"event_{event_id}" / "2d_nodes_dynamic_all.csv", nrows=1).columns.tolist()
    for c in ("rainfall", "rainfall_depth", "rainfall_intensity"):
        if c in cols:
            return c
    raise RuntimeError("rain column not found")


def build_rain_lags(rain: np.ndarray, nb: int):
    rain = np.asarray(rain, dtype=np.float32)
    if nb == 0:
        return rain[:, None]
    pad = np.concatenate([np.zeros((nb,), dtype=np.float32), rain], axis=0)
    stride = pad.strides[0]
    windows = np.lib.stride_tricks.as_strided(
        pad,
        shape=(rain.shape[0], nb + 1),
        strides=(stride, stride),
    )
    return windows[:, ::-1].copy()


def load_static_features(mdir: Path, model_id: int, node_type: int, node_ids: np.ndarray, requested_cols):
    static_path = mdir / "train" / node_static_name(node_type)
    if not static_path.exists():
        print(f"  static features not found: model={model_id} node_type={node_type} path={static_path}")
        return np.zeros((len(node_ids), 0), dtype=np.float32), []

    df = pd.read_csv(static_path)
    if "node_idx" not in df.columns:
        raise RuntimeError(f"static file missing node_idx: {static_path}")

    if requested_cols:
        cols = [c for c in requested_cols if c in df.columns and c != "node_idx"]
    else:
        cols = [c for c in df.columns if c != "node_idx" and pd.api.types.is_numeric_dtype(df[c])]

    if not cols:
        print(f"  static features empty: model={model_id} node_type={node_type}")
        return np.zeros((len(node_ids), 0), dtype=np.float32), []

    sdf = df[["node_idx"] + cols].copy()
    for c in cols:
        sdf[c] = pd.to_numeric(sdf[c], errors="coerce")
    sdf = sdf.groupby("node_idx", as_index=False).first().set_index("node_idx")
    sdf = sdf.reindex(node_ids)
    mat = sdf.to_numpy(dtype=np.float32)
    if mat.size > 0:
        col_means = np.nanmean(mat.astype(np.float64), axis=0)
        col_means = np.where(np.isfinite(col_means), col_means, 0.0).astype(np.float32)
        nan_mask = ~np.isfinite(mat)
        if nan_mask.any():
            mat = np.where(nan_mask, col_means[None, :], mat).astype(np.float32, copy=False)
        if STATIC_STANDARDIZE and mat.shape[1] > 0:
            mu = mat.mean(axis=0, dtype=np.float64).astype(np.float32)
            sd = mat.std(axis=0, dtype=np.float64).astype(np.float32)
            sd = np.where(sd < 1e-6, 1.0, sd).astype(np.float32)
            mat = ((mat - mu[None, :]) / sd[None, :]).astype(np.float32, copy=False)
    print(
        f"  static features loaded: model={model_id} node_type={node_type} "
        f"path={static_path} dim={mat.shape[1]} nodes={mat.shape[0]}"
    )
    return mat.astype(np.float32, copy=False), cols


def static_aug_dim(static1: np.ndarray, static2: np.ndarray, na: int, mode: str) -> int:
    if mode == "off" or not USE_STATIC_FEATURES_RIDGE:
        return 0
    s1 = 0 if static1.size == 0 else static1.shape[1]
    s2 = 0 if static2.size == 0 else static2.shape[1]
    if mode == "projection":
        return int(na * (s1 + s2))
    if mode == "flat":
        return int(static1.size + static2.size)
    raise ValueError(f"unknown static mode: {mode}")


def build_static_aug_train(
    y1: np.ndarray,
    y2: np.ndarray,
    start_t: int,
    na: int,
    static1: np.ndarray,
    static2: np.ndarray,
    mode: str,
) -> np.ndarray:
    sample_n = y1.shape[0] - start_t
    if sample_n <= 0 or mode == "off" or not USE_STATIC_FEATURES_RIDGE:
        return np.zeros((max(sample_n, 0), 0), dtype=np.float32)
    if mode == "flat":
        flat = []
        if static1.size > 0:
            flat.append(static1.reshape(-1).astype(np.float32, copy=False))
        if static2.size > 0:
            flat.append(static2.reshape(-1).astype(np.float32, copy=False))
        if not flat:
            return np.zeros((sample_n, 0), dtype=np.float32)
        vec = np.concatenate(flat, axis=0)
        return np.repeat(vec[None, :], sample_n, axis=0).astype(np.float32, copy=False)
    parts = []
    n1 = max(1, y1.shape[1])
    n2 = max(1, y2.shape[1])
    for k in range(na):
        hist1 = y1[start_t - 1 - k : y1.shape[0] - 1 - k, :].astype(np.float32, copy=False)
        hist2 = y2[start_t - 1 - k : y2.shape[0] - 1 - k, :].astype(np.float32, copy=False)
        if static1.size > 0:
            parts.append((hist1 @ static1 / float(n1)).astype(np.float32, copy=False))
        if static2.size > 0:
            parts.append((hist2 @ static2 / float(n2)).astype(np.float32, copy=False))
    if not parts:
        return np.zeros((sample_n, 0), dtype=np.float32)
    return np.concatenate(parts, axis=1).astype(np.float32, copy=False)


def build_static_aug_predict(
    hist1_list,
    hist2_list,
    static1: np.ndarray,
    static2: np.ndarray,
    mode: str,
) -> np.ndarray:
    if mode == "off" or not USE_STATIC_FEATURES_RIDGE:
        return np.zeros((0,), dtype=np.float32)
    if mode == "flat":
        flat = []
        if static1.size > 0:
            flat.append(static1.reshape(-1).astype(np.float32, copy=False))
        if static2.size > 0:
            flat.append(static2.reshape(-1).astype(np.float32, copy=False))
        if not flat:
            return np.zeros((0,), dtype=np.float32)
        return np.concatenate(flat, axis=0)
    parts = []
    n1 = max(1, len(hist1_list[0]) if hist1_list else static1.shape[0])
    n2 = max(1, len(hist2_list[0]) if hist2_list else static2.shape[0])
    for h1, h2 in zip(hist1_list, hist2_list):
        if static1.size > 0:
            parts.append((h1 @ static1 / float(n1)).astype(np.float32, copy=False))
        if static2.size > 0:
            parts.append((h2 @ static2 / float(n2)).astype(np.float32, copy=False))
    if not parts:
        return np.zeros((0,), dtype=np.float32)
    return np.concatenate(parts, axis=0).astype(np.float32, copy=False)


# =========================
# LightGBM path
# =========================
def load_event_target(mdir: Path, split: str, event_id: int, node_type: int, node_ids: np.ndarray, rain_col: str):
    evdir = mdir / split / f"event_{event_id}"

    if node_type == 1:
        d = pd.read_csv(evdir / "1d_nodes_dynamic_all.csv", usecols=["timestep", "node_idx", "water_level"])
        p = d.pivot(index="timestep", columns="node_idx", values="water_level").sort_index().reindex(columns=node_ids)
        r = pd.read_csv(evdir / "2d_nodes_dynamic_all.csv", usecols=["timestep", rain_col])
        rain = r.groupby("timestep")[rain_col].first().reindex(p.index).fillna(0.0)
        idx = p.index
    else:
        d = pd.read_csv(evdir / "2d_nodes_dynamic_all.csv", usecols=["timestep", "node_idx", "water_level", rain_col])
        p = d.pivot(index="timestep", columns="node_idx", values="water_level").sort_index().reindex(columns=node_ids)
        rain = d.groupby("timestep")[rain_col].first().reindex(p.index).fillna(0.0)
        idx = p.index

    return {
        "event_id": event_id,
        "timestep": idx.to_numpy(dtype=np.int32),
        "y": p.to_numpy(dtype=np.float32),
        "rain": rain.to_numpy(dtype=np.float32),
    }


def compute_node_means(events, n_nodes: int):
    s = np.zeros((n_nodes,), dtype=np.float64)
    c = np.zeros((n_nodes,), dtype=np.float64)
    for ev in events:
        y = ev["y"].astype(np.float64, copy=False)
        m = np.isfinite(y)
        s += np.where(m, y, 0.0).sum(axis=0)
        c += m.sum(axis=0)
    out = np.zeros((n_nodes,), dtype=np.float32)
    valid = c > 0
    out[valid] = (s[valid] / c[valid]).astype(np.float32, copy=False)
    if np.any(~valid):
        g = float(np.mean(out[valid])) if np.any(valid) else 0.0
        out[~valid] = g
    return out


def fill_missing(events, means):
    for ev in events:
        y = ev["y"]
        ev["y"] = np.where(np.isfinite(y), y, means[None, :]).astype(np.float32, copy=False)


def choose_time_indices(events, start_t: int, max_rows: int, n_nodes: int, seed: int):
    rows_possible = 0
    steps_each = []
    for ev in events:
        n = min(ev["y"].shape[0], ev["rain"].shape[0])
        steps = max(0, n - start_t)
        rows_possible += steps * n_nodes
        steps_each.append(steps)

    if max_rows <= 0 or rows_possible <= max_rows:
        return [None for _ in events], rows_possible, rows_possible

    ratio = float(max_rows) / float(rows_possible)
    rng = np.random.default_rng(seed)
    picks = []
    kept_rows = 0
    for steps in steps_each:
        if steps <= 0:
            picks.append(np.empty((0,), dtype=np.int32))
            continue
        keep_steps = int(round(steps * ratio))
        keep_steps = max(1, min(steps, keep_steps))
        idx = np.sort(rng.choice(steps, size=keep_steps, replace=False).astype(np.int32))
        picks.append(idx)
        kept_rows += keep_steps * n_nodes
    return picks, rows_possible, kept_rows


def build_samples_global(
    events,
    start_t: int,
    y_lags: int,
    rain_lags: int,
    max_rows: int,
    seed: int,
    return_event_ids: bool = False,
    static_mat: np.ndarray | None = None,
):
    n_nodes = events[0]["y"].shape[1]
    node_pos = np.arange(n_nodes, dtype=np.float32)

    picks, rows_possible, rows_kept = choose_time_indices(events, start_t, max_rows, n_nodes, seed)
    static_dim = 0 if static_mat is None else int(static_mat.shape[1])
    feat_dim = 1 + static_dim + y_lags + rain_lags + 1

    x = np.empty((rows_kept, feat_dim), dtype=np.float32)
    y_target = np.empty((rows_kept,), dtype=np.float32)
    row_event_ids = np.empty((rows_kept,), dtype=np.int32) if return_event_ids else None

    ptr = 0
    for j, ev in enumerate(events, 1):
        y = ev["y"]
        rain = ev["rain"]
        n = min(y.shape[0], rain.shape[0])
        if n <= start_t:
            continue
        y = y[:n]
        rain_hist = build_rain_lags(rain[:n], rain_lags)

        if picks[j - 1] is None:
            rel = np.arange(n - start_t, dtype=np.int32)
        else:
            rel = picks[j - 1]
        if rel.size == 0:
            continue

        for r in rel.tolist():
            t = start_t + int(r)
            step_y = np.stack([y[t - 1 - k, :] for k in range(y_lags)], axis=1).astype(np.float32, copy=False)
            step_r = np.tile(rain_hist[t, : rain_lags + 1][None, :], (n_nodes, 1)).astype(np.float32, copy=False)
            feat_parts = [node_pos[:, None]]
            if static_dim > 0:
                feat_parts.append(static_mat.astype(np.float32, copy=False))
            feat_parts.extend([step_y, step_r])
            feat = np.concatenate(feat_parts, axis=1)
            target = (y[t, :] - y[t - 1, :]).astype(np.float32, copy=False)

            x[ptr : ptr + n_nodes, :] = feat
            y_target[ptr : ptr + n_nodes] = target
            if return_event_ids:
                row_event_ids[ptr : ptr + n_nodes] = int(ev["event_id"])
            ptr += n_nodes

        if j % 10 == 0 or j == len(events):
            print(f"    sample progress: {j}/{len(events)} events")

    x = x[:ptr]
    y_target = y_target[:ptr]
    np.nan_to_num(x, copy=False, nan=0.0, posinf=SAFE_FEATURE_CLIP, neginf=-SAFE_FEATURE_CLIP)
    np.clip(x, -SAFE_FEATURE_CLIP, SAFE_FEATURE_CLIP, out=x)
    np.nan_to_num(y_target, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    if return_event_ids:
        row_event_ids = row_event_ids[:ptr]
        return x, y_target, rows_possible, rows_kept, row_event_ids
    return x, y_target, rows_possible, rows_kept


def fit_global_lgbm(x_train, y_train, seed: int, node_type: int, device: str, sample_weight=None):
    if node_type == 1:
        n_estimators = N_ESTIMATORS_1D
        min_child_samples = MIN_CHILD_SAMPLES_1D
    else:
        n_estimators = N_ESTIMATORS_2D
        min_child_samples = MIN_CHILD_SAMPLES_2D

    params = dict(
        objective="regression",
        n_estimators=n_estimators,
        learning_rate=LEARNING_RATE,
        num_leaves=NUM_LEAVES,
        max_depth=MAX_DEPTH,
        min_child_samples=min_child_samples,
        min_data_in_bin=MIN_DATA_IN_BIN,
        subsample=SUBSAMPLE,
        subsample_freq=1,
        colsample_bytree=COLSAMPLE_BYTREE,
        reg_alpha=REG_ALPHA,
        reg_lambda=REG_LAMBDA,
        random_state=seed,
        n_jobs=LGBM_N_JOBS,
        verbosity=-1,
        max_bin=MAX_BIN,
    )
    if device == "gpu":
        params["device_type"] = "gpu"
        params["max_cat_threshold"] = MAX_CAT_THRESHOLD
        params["gpu_use_dp"] = GPU_USE_DP

    model = LGBMRegressor(**params)
    if NODE_POS_AS_CATEGORY:
        model.fit(x_train, y_train, sample_weight=sample_weight, categorical_feature=[0])
    else:
        model.fit(x_train, y_train, sample_weight=sample_weight)
    return model


def compute_rain_event_score(ev):
    rain = np.asarray(ev["rain"], dtype=np.float64)
    rain = np.nan_to_num(rain, nan=0.0, posinf=0.0, neginf=0.0)
    total_rain = float(np.sum(np.clip(rain, 0.0, None)))
    return float(np.log1p(total_rain))


def build_rain_anchor_params(train_events):
    if not train_events:
        raise RuntimeError("train_events is empty")
    event_ids = np.asarray([int(ev["event_id"]) for ev in train_events], dtype=np.int32)
    scores = np.asarray([compute_rain_event_score(ev) for ev in train_events], dtype=np.float64)
    q = np.quantile(scores, [RAIN_ANCHOR_QLOW, RAIN_ANCHOR_QMID, RAIN_ANCHOR_QHIGH]).astype(np.float64)
    centers = q.astype(np.float32, copy=False)
    spread = float(max(q[2] - q[0], RAIN_ANCHOR_MIN_BW))
    bandwidth = float(max(RAIN_ANCHOR_BW_SCALE * spread, RAIN_ANCHOR_MIN_BW))
    score_by_event = {int(eid): float(sc) for eid, sc in zip(event_ids.tolist(), scores.tolist())}
    return {
        "centers": centers,
        "bandwidth": bandwidth,
        "score_by_event": score_by_event,
    }


def anchor_raw_weights(score: float, centers: np.ndarray, bandwidth: float):
    d = np.abs(np.asarray(score, dtype=np.float64) - centers.astype(np.float64))
    raw = np.maximum(0.0, 1.0 - d / float(bandwidth)).astype(np.float32, copy=False)
    if float(np.sum(raw)) <= 0.0:
        raw[:] = 0.0
        raw[int(np.argmin(d))] = 1.0
    return raw


def anchor_norm_weights(score: float, centers: np.ndarray, bandwidth: float):
    raw = anchor_raw_weights(score, centers, bandwidth)
    s = float(np.sum(raw))
    if s <= 0.0:
        return np.array([1.0, 0.0, 0.0], dtype=np.float32)
    return (raw / s).astype(np.float32, copy=False)


def build_anchor_sample_weights(row_event_ids: np.ndarray, score_by_event: dict, centers: np.ndarray, bandwidth: float):
    row_event_ids = np.asarray(row_event_ids, dtype=np.int32)
    if row_event_ids.size == 0:
        raise RuntimeError("row_event_ids is empty")
    uniq_ids, inv = np.unique(row_event_ids, return_inverse=True)
    uniq_raw = np.empty((uniq_ids.size, centers.size), dtype=np.float32)
    for i, eid in enumerate(uniq_ids.tolist()):
        score = float(score_by_event.get(int(eid), 0.0))
        uniq_raw[i, :] = anchor_raw_weights(score, centers, bandwidth)
    return uniq_raw[inv, :].astype(np.float32, copy=False)


def predict_event_recursive(
    ev,
    model: LGBMRegressor,
    start_t: int,
    y_lags: int,
    rain_lags: int,
    static_mat: np.ndarray | None = None,
):
    y_true = ev["y"]
    rain = ev["rain"]
    ts = ev["timestep"]
    n = min(y_true.shape[0], rain.shape[0], ts.shape[0])
    y_true = y_true[:n]
    rain = rain[:n]
    ts = ts[:n]

    n_nodes = y_true.shape[1]
    node_pos = np.arange(n_nodes, dtype=np.float32)
    rain_hist = build_rain_lags(rain, rain_lags)
    static_dim = 0 if static_mat is None else int(static_mat.shape[1])

    pred = y_true.copy()
    feat = np.empty((n_nodes, 1 + static_dim + y_lags + rain_lags + 1), dtype=np.float32)
    booster = model.booster_

    for t in range(start_t, n):
        feat[:, 0] = node_pos
        offset = 1
        if static_dim > 0:
            feat[:, offset : offset + static_dim] = static_mat
            offset += static_dim
        for k in range(y_lags):
            feat[:, offset + k] = pred[t - 1 - k, :]
        feat[:, offset + y_lags :] = rain_hist[t, : rain_lags + 1][None, :]

        np.nan_to_num(feat, copy=False, nan=0.0, posinf=SAFE_FEATURE_CLIP, neginf=-SAFE_FEATURE_CLIP)
        np.clip(feat, -SAFE_FEATURE_CLIP, SAFE_FEATURE_CLIP, out=feat)

        delta = booster.predict(feat).astype(np.float32, copy=False)
        np.nan_to_num(delta, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        pred[t, :] = pred[t - 1, :] + delta
        np.nan_to_num(pred[t, :], copy=False, nan=0.0, posinf=SAFE_LEVEL_CLIP, neginf=-SAFE_LEVEL_CLIP)
        np.clip(pred[t, :], -SAFE_LEVEL_CLIP, SAFE_LEVEL_CLIP, out=pred[t, :])

    return {
        "timestep": ts,
        "true": y_true,
        "pred": pred,
        "n": n,
    }


def predict_event_recursive_weighted(
    ev,
    models,
    start_t: int,
    y_lags: int,
    rain_lags: int,
    anchor_params,
    static_mat: np.ndarray | None = None,
):
    if len(models) != 3:
        raise RuntimeError(f"weighted mode expects 3 anchor models, got {len(models)}")
    score = compute_rain_event_score(ev)
    weights = anchor_norm_weights(score, anchor_params["centers"], anchor_params["bandwidth"])
    outs = [predict_event_recursive(ev, m, start_t, y_lags, rain_lags, static_mat=static_mat) for m in models]
    base = outs[0]
    pred = np.zeros_like(base["pred"], dtype=np.float32)
    for wk, out in zip(weights.tolist(), outs):
        pred += float(wk) * out["pred"].astype(np.float32, copy=False)
    return {
        "timestep": base["timestep"],
        "true": base["true"],
        "pred": pred,
        "n": base["n"],
        "rain_score": float(score),
        "anchor_weights": weights,
    }


def evaluate_target_lgbm_per_node(
    model_id: int,
    node_type: int,
    val_events,
    model,
    start_t: int,
    y_lags: int,
    rain_lags: int,
    static_mat: np.ndarray | None = None,
):
    std = float(STD_DEV[(model_id, node_type)])
    n_nodes = None
    se_sum = None
    cnt_sum = None

    for j, ev in enumerate(val_events, 1):
        out = predict_event_recursive(ev, model, start_t, y_lags, rain_lags, static_mat=static_mat)
        n = out["n"]
        true = out["true"][start_t:n, :]
        pred = out["pred"][start_t:n, :]
        if true.size == 0:
            continue

        if n_nodes is None:
            n_nodes = true.shape[1]
            se_sum = np.zeros((n_nodes,), dtype=np.float64)
            cnt_sum = np.zeros((n_nodes,), dtype=np.float64)

        err2 = (pred - true) ** 2
        se_sum += np.sum(err2, axis=0, dtype=np.float64)
        cnt_sum += float(true.shape[0])

        if j % 5 == 0 or j == len(val_events):
            print(f"    val progress model {model_id} node_type {node_type}: {j}/{len(val_events)} events")

    if n_nodes is None:
        return float("nan"), np.array([], dtype=np.float32)

    mse = se_sum / np.maximum(cnt_sum, 1.0)
    rmse = np.sqrt(mse) / std
    rmse = rmse.astype(np.float32, copy=False)
    return float(np.mean(rmse)), rmse


def evaluate_target_lgbm_per_node_weighted(
    model_id: int,
    node_type: int,
    val_events,
    models,
    start_t: int,
    y_lags: int,
    rain_lags: int,
    anchor_params,
    static_mat: np.ndarray | None = None,
):
    std = float(STD_DEV[(model_id, node_type)])
    n_nodes = None
    se_sum = None
    cnt_sum = None

    for j, ev in enumerate(val_events, 1):
        out = predict_event_recursive_weighted(
            ev,
            models=models,
            start_t=start_t,
            y_lags=y_lags,
            rain_lags=rain_lags,
            anchor_params=anchor_params,
            static_mat=static_mat,
        )
        n = out["n"]
        true = out["true"][start_t:n, :]
        pred = out["pred"][start_t:n, :]
        if true.size == 0:
            continue

        if n_nodes is None:
            n_nodes = true.shape[1]
            se_sum = np.zeros((n_nodes,), dtype=np.float64)
            cnt_sum = np.zeros((n_nodes,), dtype=np.float64)

        err2 = (pred - true) ** 2
        se_sum += np.sum(err2, axis=0, dtype=np.float64)
        cnt_sum += float(true.shape[0])

        if j % 5 == 0 or j == len(val_events):
            print(f"    val progress weighted model {model_id} node_type {node_type}: {j}/{len(val_events)} events")

    if n_nodes is None:
        return float("nan"), np.array([], dtype=np.float32)

    mse = se_sum / np.maximum(cnt_sum, 1.0)
    rmse = np.sqrt(mse) / std
    rmse = rmse.astype(np.float32, copy=False)
    return float(np.mean(rmse)), rmse


def predict_target_df_lgbm(
    model_id: int,
    node_type: int,
    test_events,
    node_ids,
    model,
    start_t: int,
    y_lags: int,
    rain_lags: int,
    selected_node_ids=None,
    static_mat: np.ndarray | None = None,
):
    node_ids = np.asarray(node_ids, dtype=np.int32)
    if selected_node_ids is None:
        use_node_ids = node_ids
        use_idx = None
    else:
        selected_node_ids = np.asarray(selected_node_ids, dtype=np.int32)
        if selected_node_ids.size == 0:
            return pd.DataFrame(columns=KEY_COLS + ["water_level"])
        idx = np.searchsorted(node_ids, selected_node_ids)
        valid = (idx >= 0) & (idx < node_ids.size) & (node_ids[idx] == selected_node_ids)
        if not np.all(valid):
            missing = selected_node_ids[~valid][:10].tolist()
            raise RuntimeError(f"selected_node_ids not found in node_ids (first 10): {missing}")
        use_node_ids = selected_node_ids
        use_idx = idx

    rows = []
    total = len(test_events)
    for j, ev in enumerate(test_events, 1):
        out = predict_event_recursive(ev, model, start_t, y_lags, rain_lags, static_mat=static_mat)
        n = out["n"]
        ts = out["timestep"][start_t:n]
        pred = out["pred"][start_t:n, :]
        if use_idx is not None:
            pred = pred[:, use_idx]

        if ts.size == 0:
            continue

        n_rows = ts.size * len(use_node_ids)
        part = pd.DataFrame(
            {
                "model_id": np.full((n_rows,), model_id, dtype=np.int8),
                "event_id": np.full((n_rows,), int(ev["event_id"]), dtype=np.int32),
                "node_type": np.full((n_rows,), node_type, dtype=np.int8),
                "node_id": np.tile(use_node_ids, ts.size),
                "timestep": np.repeat(ts, len(use_node_ids)),
                "water_level": pred.reshape(-1).astype(np.float32, copy=False),
            }
        )
        rows.append(part)
        if j % 5 == 0 or j == total:
            print(f"    predict progress model {model_id} node_type {node_type}: {j}/{total} test events")

    if not rows:
        return pd.DataFrame(columns=KEY_COLS + ["water_level"])
    return pd.concat(rows, axis=0, ignore_index=True)


def predict_target_df_lgbm_weighted(
    model_id: int,
    node_type: int,
    test_events,
    node_ids,
    models,
    start_t: int,
    y_lags: int,
    rain_lags: int,
    anchor_params,
    selected_node_ids=None,
    static_mat: np.ndarray | None = None,
):
    node_ids = np.asarray(node_ids, dtype=np.int32)
    if selected_node_ids is None:
        use_node_ids = node_ids
        use_idx = None
    else:
        selected_node_ids = np.asarray(selected_node_ids, dtype=np.int32)
        if selected_node_ids.size == 0:
            return pd.DataFrame(columns=KEY_COLS + ["water_level"])
        idx = np.searchsorted(node_ids, selected_node_ids)
        valid = (idx >= 0) & (idx < node_ids.size) & (node_ids[idx] == selected_node_ids)
        if not np.all(valid):
            missing = selected_node_ids[~valid][:10].tolist()
            raise RuntimeError(f"selected_node_ids not found in node_ids (first 10): {missing}")
        use_node_ids = selected_node_ids
        use_idx = idx

    rows = []
    total = len(test_events)
    for j, ev in enumerate(test_events, 1):
        out = predict_event_recursive_weighted(
            ev,
            models=models,
            start_t=start_t,
            y_lags=y_lags,
            rain_lags=rain_lags,
            anchor_params=anchor_params,
            static_mat=static_mat,
        )
        n = out["n"]
        ts = out["timestep"][start_t:n]
        pred = out["pred"][start_t:n, :]
        if use_idx is not None:
            pred = pred[:, use_idx]

        if ts.size == 0:
            continue

        n_rows = ts.size * len(use_node_ids)
        part = pd.DataFrame(
            {
                "model_id": np.full((n_rows,), model_id, dtype=np.int8),
                "event_id": np.full((n_rows,), int(ev["event_id"]), dtype=np.int32),
                "node_type": np.full((n_rows,), node_type, dtype=np.int8),
                "node_id": np.tile(use_node_ids, ts.size),
                "timestep": np.repeat(ts, len(use_node_ids)),
                "water_level": pred.reshape(-1).astype(np.float32, copy=False),
            }
        )
        rows.append(part)
        if j % 5 == 0 or j == total:
            print(f"    predict progress model {model_id} node_type {node_type}: {j}/{total} test events")

    if not rows:
        return pd.DataFrame(columns=KEY_COLS + ["water_level"])
    return pd.concat(rows, axis=0, ignore_index=True)


# =========================
# Joint ridge path
# =========================
def load_node_pivot(mdir: Path, split: str, event_id: int, node_type: int, node_ids):
    fpath = mdir / split / f"event_{event_id}" / node_dynamic_name(node_type)
    if not fpath.exists():
        return None
    df = pd.read_csv(fpath, usecols=["timestep", "node_idx", "water_level"])
    if df.empty:
        return None
    pivot = df.pivot(index="timestep", columns="node_idx", values="water_level").sort_index()
    pivot = pivot.reindex(columns=node_ids)
    return pivot


def load_rain_series(mdir: Path, split: str, event_id: int):
    fpath = mdir / split / f"event_{event_id}" / "2d_nodes_dynamic_all.csv"
    if not fpath.exists():
        raise FileNotFoundError(f"Rainfall source not found: {fpath}")
    df = pd.read_csv(fpath)
    rain_col = None
    for c in ("rainfall", "rainfall_depth", "rainfall_intensity"):
        if c in df.columns:
            rain_col = c
            break
    if rain_col is None:
        raise ValueError(f"Rainfall column not found in {fpath}")
    rain = df.groupby("timestep")[rain_col].first().sort_index()
    return rain


def load_joint_event(mdir: Path, split: str, event_id: int, node_ids_1, node_ids_2, nb: int):
    p1 = load_node_pivot(mdir, split, event_id, 1, node_ids_1)
    p2 = load_node_pivot(mdir, split, event_id, 2, node_ids_2)
    rain = load_rain_series(mdir, split, event_id)

    idx = rain.index
    if p1 is not None:
        idx = idx.union(p1.index)
    if p2 is not None:
        idx = idx.union(p2.index)
    idx = idx.sort_values()
    if len(idx) == 0:
        return None

    t = idx.to_numpy(dtype=np.int32)

    if p1 is None:
        y1 = np.full((len(t), len(node_ids_1)), np.nan, dtype=np.float32)
    else:
        y1 = p1.reindex(idx).to_numpy(dtype=np.float32)

    if p2 is None:
        y2 = np.full((len(t), len(node_ids_2)), np.nan, dtype=np.float32)
    else:
        y2 = p2.reindex(idx).to_numpy(dtype=np.float32)

    rain_vals = rain.reindex(idx, fill_value=0.0).to_numpy(dtype=np.float32)
    rain_lags = build_rain_lags(rain_vals, nb)
    return t, y1, y2, rain_lags


def compute_node_means_from_csv(mdir: Path, node_type: int, node_ids, train_events):
    sums = np.zeros((len(node_ids),), dtype=np.float64)
    counts = np.zeros((len(node_ids),), dtype=np.int64)

    for event_id in train_events:
        fpath = mdir / "train" / f"event_{event_id}" / node_dynamic_name(node_type)
        if not fpath.exists():
            continue
        df = pd.read_csv(fpath, usecols=["node_idx", "water_level"]).dropna()
        if df.empty:
            continue
        agg = df.groupby("node_idx")["water_level"].agg(["sum", "count"])
        ids = agg.index.to_numpy(dtype=np.int32)
        idx = np.searchsorted(node_ids, ids)
        valid = (idx < node_ids.shape[0]) & (node_ids[idx] == ids)
        if not np.any(valid):
            continue
        tgt = idx[valid]
        sums[tgt] += agg["sum"].to_numpy(dtype=np.float64)[valid]
        counts[tgt] += agg["count"].to_numpy(dtype=np.int64)[valid]

    mean = sums / np.maximum(counts, 1)
    has = counts > 0
    global_mean = float(np.mean(mean[has])) if np.any(has) else 0.0
    mean[~has] = global_mean
    return mean.astype(np.float32)


def build_normal_equations_joint(
    mdir: Path,
    node_ids_1,
    node_ids_2,
    node_means_1,
    node_means_2,
    static1: np.ndarray,
    static2: np.ndarray,
    train_events,
    na: int,
    nb: int,
    static_mode: str,
):
    n1 = len(node_ids_1)
    n2 = len(node_ids_2)
    n_total = n1 + n2

    feat_dim = na * n_total + (nb + 1) + static_aug_dim(static1, static2, na, static_mode) + 1
    xtx = np.zeros((feat_dim, feat_dim), dtype=np.float64)
    xty = np.zeros((feat_dim, n_total), dtype=np.float64)

    n_rows = 0
    used_events = 0

    for j, event_id in enumerate(train_events, 1):
        loaded = load_joint_event(mdir, "train", event_id, node_ids_1, node_ids_2, nb)
        if loaded is None:
            continue

        _, y1, y2, rain_lags = loaded
        n = min(y1.shape[0], y2.shape[0], rain_lags.shape[0])
        if n <= START_T:
            continue

        y1 = np.where(np.isfinite(y1), y1, node_means_1[None, :]).astype(np.float32, copy=False)
        y2 = np.where(np.isfinite(y2), y2, node_means_2[None, :]).astype(np.float32, copy=False)
        y = np.concatenate([y1[:n], y2[:n]], axis=1)
        np.clip(y, -SAFE_LEVEL_CLIP, SAFE_LEVEL_CLIP, out=y)

        rain_lags = rain_lags[:n, : nb + 1]
        sample_n = n - START_T
        lag_blocks = [y[START_T - 1 - k : n - 1 - k, :] for k in range(na)]
        x_parts = lag_blocks + [rain_lags[START_T:n, :].astype(np.float32, copy=False)]
        static_aug = build_static_aug_train(y1, y2, START_T, na, static1, static2, static_mode)
        if static_aug.shape[1] > 0:
            x_parts.append(static_aug)
        x_parts.append(np.ones((sample_n, 1), dtype=np.float32))
        x = np.concatenate(x_parts, axis=1).astype(np.float64, copy=False)

        y_delta = (y[START_T:n, :] - y[START_T - 1 : n - 1, :]).astype(np.float64, copy=False)
        np.clip(y_delta, -SAFE_DELTA_CLIP, SAFE_DELTA_CLIP, out=y_delta)

        xtx += x.T @ x
        xty += x.T @ y_delta
        n_rows += sample_n
        used_events += 1

        if j % 10 == 0 or j == len(train_events):
            print(f"    train progress: {j}/{len(train_events)} events")

    return xtx, xty, n_rows, used_events


def fit_joint_ridge(
    mdir: Path,
    model_id: int,
    node_ids_1,
    node_ids_2,
    node_means_1,
    node_means_2,
    static1: np.ndarray,
    static2: np.ndarray,
    train_events,
    na: int,
    nb: int,
    alpha: float,
    static_mode: str,
):
    xtx, xty, n_rows, used_events = build_normal_equations_joint(
        mdir,
        node_ids_1,
        node_ids_2,
        node_means_1,
        node_means_2,
        static1,
        static2,
        train_events,
        na,
        nb,
        static_mode,
    )
    if n_rows == 0:
        raise RuntimeError(f"No valid train rows for model={model_id}")

    reg = alpha * np.eye(xtx.shape[0], dtype=np.float64)
    reg[-1, -1] = 0.0
    a = xtx + reg
    try:
        w = np.linalg.solve(a, xty)
    except np.linalg.LinAlgError:
        w = np.linalg.lstsq(a, xty, rcond=None)[0]

    return {
        "model_id": model_id,
        "mdir": mdir,
        "na": int(na),
        "nb": int(nb),
        "alpha": float(alpha),
        "w": w.astype(np.float32),
        "node_ids_1": node_ids_1,
        "node_ids_2": node_ids_2,
        "node_means_1": node_means_1,
        "node_means_2": node_means_2,
        "n1": int(len(node_ids_1)),
        "n2": int(len(node_ids_2)),
        "static1": static1.astype(np.float32, copy=False),
        "static2": static2.astype(np.float32, copy=False),
        "static_mode": static_mode,
        "train_rows": int(n_rows),
        "used_events": int(used_events),
    }


def predict_joint_event_recursive(state, split: str, event_id: int):
    loaded = load_joint_event(
        state["mdir"],
        split,
        event_id,
        state["node_ids_1"],
        state["node_ids_2"],
        state["nb"],
    )
    if loaded is None:
        return None

    t, y1, y2, rain_lags = loaded
    n = min(y1.shape[0], y2.shape[0], rain_lags.shape[0])
    if n <= START_T:
        return None

    y1 = np.where(np.isfinite(y1[:n]), y1[:n], state["node_means_1"][None, :]).astype(np.float32, copy=False)
    y2 = np.where(np.isfinite(y2[:n]), y2[:n], state["node_means_2"][None, :]).astype(np.float32, copy=False)
    y = np.concatenate([y1, y2], axis=1)
    rain_lags = rain_lags[:n, : state["nb"] + 1]
    np.clip(y, -SAFE_LEVEL_CLIP, SAFE_LEVEL_CLIP, out=y)

    preds = y.copy()
    w = state["w"]
    na = state["na"]
    nb = state["nb"]
    n1 = state["n1"]
    n2 = state["n2"]

    for tt in range(START_T, n):
        hist1_list = [preds[tt - 1 - k, :n1] for k in range(na)]
        hist2_list = [preds[tt - 1 - k, n1:] for k in range(na)]
        lag_parts = [preds[tt - 1 - k, :] for k in range(na)]
        x_parts = lag_parts + [rain_lags[tt, : nb + 1].astype(np.float32, copy=False)]
        static_vec = build_static_aug_predict(
            hist1_list,
            hist2_list,
            state["static1"],
            state["static2"],
            state["static_mode"],
        )
        if static_vec.size > 0:
            x_parts.append(static_vec)
        x_parts.append(np.array([1.0], dtype=np.float32))
        x = np.concatenate(x_parts, axis=0).astype(np.float32, copy=False)
        delta = x @ w
        np.nan_to_num(delta, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        np.clip(delta, -SAFE_DELTA_CLIP, SAFE_DELTA_CLIP, out=delta)
        preds[tt, :] = preds[tt - 1, :] + delta
        np.nan_to_num(preds[tt, :], copy=False, nan=0.0, posinf=SAFE_LEVEL_CLIP, neginf=-SAFE_LEVEL_CLIP)
        np.clip(preds[tt, :], -SAFE_LEVEL_CLIP, SAFE_LEVEL_CLIP, out=preds[tt, :])

    t_pred = t[START_T:n]
    pred = preds[START_T:n, :]
    y_true = y[START_T:n, :]

    pred1 = pred[:, :n1]
    pred2 = pred[:, n1:]
    y1_true = y_true[:, :n1]
    y2_true = y_true[:, n1:]
    return t_pred, pred1, pred2, y1_true, y2_true


def evaluate_model_ridge_event_average(state, val_events, fold_idx=None, n_folds=None):
    model_id = state["model_id"]
    event_scores_by_type = {1: [], 2: []}
    has_data = False

    for j, event_id in enumerate(val_events, 1):
        out = predict_joint_event_recursive(state, "train", event_id)
        if out is None:
            continue
        _, pred1, pred2, y1_true, y2_true = out
        if pred1.size > 0:
            err1 = pred1 - y1_true
            rmse1 = np.sqrt(np.mean(err1 * err1, axis=0))
            score1 = float(np.mean(rmse1 / float(STD_DEV[(model_id, 1)])))
            event_scores_by_type[1].append(score1)
            has_data = True
        if pred2.size > 0:
            err2 = pred2 - y2_true
            rmse2 = np.sqrt(np.mean(err2 * err2, axis=0))
            score2 = float(np.mean(rmse2 / float(STD_DEV[(model_id, 2)])))
            event_scores_by_type[2].append(score2)
            has_data = True

        if j % 5 == 0 or j == len(val_events):
            msg = []
            for node_type in (1, 2):
                vals = event_scores_by_type[node_type]
                if vals:
                    msg.append(f"type={node_type} running_std_rmse={float(np.mean(vals)):.6f}")
            prefix = f"    cv progress model={model_id} "
            if fold_idx is not None and n_folds is not None:
                prefix += f"fold={fold_idx + 1}/{n_folds} "
            print(prefix + f"events={j}/{len(val_events)} {' '.join(msg)}")

    if not has_data:
        return None, None, np.array([], dtype=np.float32), np.array([], dtype=np.float32)

    s1 = float(np.mean(event_scores_by_type[1])) if event_scores_by_type[1] else float("nan")
    s2 = float(np.mean(event_scores_by_type[2])) if event_scores_by_type[2] else float("nan")
    return s1, s2, np.asarray(event_scores_by_type[1], dtype=np.float32), np.asarray(event_scores_by_type[2], dtype=np.float32)


def build_ridge_val_pred_index(state, node_type: int, val_event_ids):
    pred_by_event = {}
    for event_id in val_event_ids:
        out = predict_joint_event_recursive(state, "train", event_id)
        if out is None:
            continue
        t_pred, pred1, pred2, _, _ = out
        pred = pred1 if node_type == 1 else pred2
        pred_by_event[int(event_id)] = (t_pred.astype(np.int32, copy=False), pred.astype(np.float32, copy=False))
    return pred_by_event


def compute_lambda_sse_per_node(target, ridge_state, lgbm_st, val_event_ids, return_oof_part: bool = False):
    model_id, node_type = target
    node_ids = np.asarray(lgbm_st["node_ids"], dtype=np.int32)
    n_nodes = len(node_ids)

    ridge_val_pred = build_ridge_val_pred_index(ridge_state, node_type, val_event_ids)
    sse0 = np.zeros((n_nodes,), dtype=np.float64)
    sse1 = np.zeros((n_nodes,), dtype=np.float64)
    sse2 = np.zeros((n_nodes,), dtype=np.float64)
    cnt = np.zeros((n_nodes,), dtype=np.float64)
    used_events = 0
    oof_parts = [] if return_oof_part else None

    for j, ev in enumerate(lgbm_st["val_events"], 1):
        event_id = int(ev["event_id"])
        if event_id not in ridge_val_pred:
            continue

        out_l = predict_event_recursive_weighted(
            ev,
            models=lgbm_st["models"],
            start_t=START_T,
            y_lags=lgbm_st["y_lags"],
            rain_lags=lgbm_st["rain_lags"],
            anchor_params=lgbm_st["anchor_params"],
            static_mat=lgbm_st.get("static_mat"),
        )
        n_l = out_l["n"]
        ts_l = out_l["timestep"][START_T:n_l]
        true_l = out_l["true"][START_T:n_l, :].astype(np.float32, copy=False)
        pred_l = out_l["pred"][START_T:n_l, :].astype(np.float32, copy=False)
        if ts_l.size == 0:
            continue

        ts_r, pred_r = ridge_val_pred[event_id]
        if ts_r.size == 0:
            continue

        if ts_l.shape == ts_r.shape and np.array_equal(ts_l, ts_r):
            true_use = true_l
            pred_l_use = pred_l
            pred_r_use = pred_r
            ts_use = ts_l.astype(np.int32, copy=False)
            t_used = ts_l.size
        else:
            ts_common, idx_l, idx_r = np.intersect1d(ts_l, ts_r, return_indices=True)
            if ts_common.size == 0:
                continue
            true_use = true_l[idx_l, :]
            pred_l_use = pred_l[idx_l, :]
            pred_r_use = pred_r[idx_r, :]
            ts_use = ts_common.astype(np.int32, copy=False)
            t_used = ts_common.size

        base = pred_r_use - true_use
        diff = pred_l_use - pred_r_use
        np.nan_to_num(base, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        np.nan_to_num(diff, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

        base64 = base.astype(np.float64, copy=False)
        diff64 = diff.astype(np.float64, copy=False)
        sse0 += np.sum(base64 * base64, axis=0)
        sse1 += 2.0 * np.sum(base64 * diff64, axis=0)
        sse2 += np.sum(diff64 * diff64, axis=0)
        cnt += float(t_used)
        used_events += 1

        if return_oof_part:
            n_rows_event = int(t_used * n_nodes)
            oof_parts.append(
                pd.DataFrame(
                    {
                        "model_id": np.full((n_rows_event,), model_id, dtype=np.int8),
                        "event_id": np.full((n_rows_event,), event_id, dtype=np.int32),
                        "node_type": np.full((n_rows_event,), node_type, dtype=np.int8),
                        "node_id": np.tile(node_ids, t_used),
                        "timestep": np.repeat(ts_use, n_nodes),
                        "water_level_true": true_use.reshape(-1).astype(np.float32, copy=False),
                        "ridge_pred": pred_r_use.reshape(-1).astype(np.float32, copy=False),
                        "lgbm_pred": pred_l_use.reshape(-1).astype(np.float32, copy=False),
                    }
                )
            )

        if j % 5 == 0 or j == len(lgbm_st["val_events"]):
            print(f"    val blend progress model {model_id} node_type {node_type}: {j}/{len(lgbm_st['val_events'])} events")

    oof_df = None
    if return_oof_part:
        if oof_parts:
            oof_df = pd.concat(oof_parts, ignore_index=True)
            oof_df = oof_df.sort_values(KEY_COLS).reset_index(drop=True)
        else:
            oof_df = pd.DataFrame(columns=KEY_COLS + ["water_level_true", "ridge_pred", "lgbm_pred"])
    return sse0, sse1, sse2, cnt, used_events, oof_df


def summarize_lambda_from_sse(target, sse0, sse1, sse2, cnt, used_events: int):
    std = float(STD_DEV[target])
    n_nodes = sse0.shape[0]

    if used_events == 0 or n_nodes == 0:
        nan_arr = np.full((n_nodes,), np.nan, dtype=np.float32)
        return {
            "ridge_mean": float("nan"),
            "lgbm_mean": float("nan"),
            "selected_mean": float("nan"),
            "ridge_node_rmse": nan_arr.copy(),
            "lgbm_node_rmse": nan_arr.copy(),
            "selected_node_rmse": nan_arr.copy(),
            "best_lambda": np.zeros((n_nodes,), dtype=np.float32),
            "used_events": 0,
        }

    cnt_safe = np.maximum(cnt, 1.0)
    ridge_node_rmse = (np.sqrt(np.maximum(sse0, 0.0) / cnt_safe) / std).astype(np.float32, copy=False)
    lgbm_node_rmse = (np.sqrt(np.maximum(sse0 + sse1 + sse2, 0.0) / cnt_safe) / std).astype(np.float32, copy=False)

    if FORCE_METHOD == "ridge":
        lam = np.zeros((n_nodes,), dtype=np.float64)
    elif FORCE_METHOD == "lgbm":
        lam = np.ones((n_nodes,), dtype=np.float64)
    else:
        lam = np.zeros((n_nodes,), dtype=np.float64)
        denom = 2.0 * sse2
        mask = sse2 > LAMBDA_CLIP_EPS
        lam[mask] = -sse1[mask] / denom[mask]
        boundary = (~mask) & (sse1 < 0.0)
        lam[boundary] = 1.0
        np.clip(lam, 0.0, 1.0, out=lam)

    sel_sse = sse0 + sse1 * lam + sse2 * lam * lam
    selected_node_rmse = (np.sqrt(np.maximum(sel_sse, 0.0) / cnt_safe) / std).astype(np.float32, copy=False)
    best_lambda = lam.astype(np.float32, copy=False)

    return {
        "ridge_mean": float(np.nanmean(ridge_node_rmse)),
        "lgbm_mean": float(np.nanmean(lgbm_node_rmse)),
        "selected_mean": float(np.nanmean(selected_node_rmse)),
        "ridge_node_rmse": ridge_node_rmse,
        "lgbm_node_rmse": lgbm_node_rmse,
        "selected_node_rmse": selected_node_rmse,
        "best_lambda": best_lambda,
        "used_events": used_events,
    }


def build_model_prediction_df_ridge(state, split: str = "test", node_type: int | None = None, selected_node_ids=None, event_ids=None):
    model_id = state["model_id"]
    if node_type not in (None, 1, 2):
        raise ValueError(f"node_type must be None, 1, or 2, got {node_type}")

    idx1 = None
    idx2 = None
    use_node_ids_1 = state["node_ids_1"]
    use_node_ids_2 = state["node_ids_2"]
    if selected_node_ids is not None:
        selected_node_ids = np.asarray(selected_node_ids, dtype=np.int32)
        if selected_node_ids.size == 0:
            return pd.DataFrame(columns=KEY_COLS + ["water_level"])
        if node_type == 1:
            idx1 = np.searchsorted(state["node_ids_1"], selected_node_ids)
            valid1 = (idx1 >= 0) & (idx1 < state["node_ids_1"].size) & (state["node_ids_1"][idx1] == selected_node_ids)
            if not np.all(valid1):
                missing = selected_node_ids[~valid1][:10].tolist()
                raise RuntimeError(f"selected ridge node_ids(type=1) not found (first 10): {missing}")
            use_node_ids_1 = selected_node_ids
        elif node_type == 2:
            idx2 = np.searchsorted(state["node_ids_2"], selected_node_ids)
            valid2 = (idx2 >= 0) & (idx2 < state["node_ids_2"].size) & (state["node_ids_2"][idx2] == selected_node_ids)
            if not np.all(valid2):
                missing = selected_node_ids[~valid2][:10].tolist()
                raise RuntimeError(f"selected ridge node_ids(type=2) not found (first 10): {missing}")
            use_node_ids_2 = selected_node_ids
        else:
            raise ValueError("selected_node_ids requires explicit node_type")

    parts = []
    if event_ids is None:
        event_ids = list_event_ids(state["mdir"], split)
    else:
        event_ids = [int(x) for x in event_ids]

    for j, event_id in enumerate(event_ids, 1):
        out = predict_joint_event_recursive(state, split, event_id)
        if out is None:
            continue

        t_pred, pred1, pred2, _, _ = out
        n_t = len(t_pred)

        if node_type in (None, 1):
            p1 = pred1 if idx1 is None else pred1[:, idx1]
            n1 = p1.shape[1]
            n_rows1 = n_t * n1
            frame1 = pd.DataFrame(
                {
                    "model_id": np.full((n_rows1,), model_id, dtype=np.int8),
                    "event_id": np.full((n_rows1,), event_id, dtype=np.int32),
                    "node_type": np.full((n_rows1,), 1, dtype=np.int8),
                    "node_id": np.tile(use_node_ids_1, n_t),
                    "timestep": np.repeat(t_pred.astype(np.int32), n1),
                    "water_level": p1.reshape(-1).astype(np.float32, copy=False),
                }
            )
            parts.append(frame1)

        if node_type in (None, 2):
            p2 = pred2 if idx2 is None else pred2[:, idx2]
            n2 = p2.shape[1]
            n_rows2 = n_t * n2
            frame2 = pd.DataFrame(
                {
                    "model_id": np.full((n_rows2,), model_id, dtype=np.int8),
                    "event_id": np.full((n_rows2,), event_id, dtype=np.int32),
                    "node_type": np.full((n_rows2,), 2, dtype=np.int8),
                    "node_id": np.tile(use_node_ids_2, n_t),
                    "timestep": np.repeat(t_pred.astype(np.int32), n2),
                    "water_level": p2.reshape(-1).astype(np.float32, copy=False),
                }
            )
            parts.append(frame2)

        if j % 5 == 0 or j == len(event_ids):
            print(f"    predict progress ridge model {model_id}: {j}/{len(event_ids)} events")

    if not parts:
        return pd.DataFrame(columns=KEY_COLS + ["water_level"])
    return pd.concat(parts, ignore_index=True)


def model_ridge_params(model_id: int):
    na = NA_MODEL1 if model_id == 1 else NA_MODEL2
    nb = NB_MODEL1 if model_id == 1 else NB_MODEL2
    return na, nb, RIDGE_ALPHA


def format_score(v):
    if v is None:
        return "None"
    if isinstance(v, float) and np.isnan(v):
        return "nan"
    return f"{float(v):.6f}"


def load_events_for_ids(mdir: Path, split: str, event_ids, node_type: int, node_ids: np.ndarray, rain_col: str):
    return [load_event_target(mdir, split, eid, node_type, node_ids, rain_col) for eid in event_ids]


def align_prediction_vector(base_target: pd.DataFrame, pred_df: pd.DataFrame, has_key_cols: bool):
    if has_key_cols:
        pred_use = pred_df[KEY_COLS + ["water_level"]]
        merged = base_target.merge(pred_use, on=KEY_COLS, how="left", validate="one_to_one", sort=False)
    else:
        base_keys = ["model_id", "event_id", "node_type", "node_id"]
        pred_df = pred_df.sort_values(KEY_COLS).copy()
        pred_df["__step_idx"] = pred_df.groupby(base_keys).cumcount()
        pred_use = pred_df[base_keys + ["__step_idx", "water_level"]]
        merged = base_target.merge(
            pred_use,
            on=base_keys + ["__step_idx"],
            how="left",
            validate="one_to_one",
            sort=False,
        )
    vals = merged["water_level"].to_numpy(dtype=np.float32, copy=False)
    if np.isnan(vals).any():
        miss = int(np.isnan(vals).sum())
        raise RuntimeError(f"prediction alignment produced missing rows: {miss}")
    return vals


def lookup_sorted_positions(ref_ids, query_ids, context: str):
    ref_ids = np.asarray(ref_ids, dtype=np.int32)
    query_ids = np.asarray(query_ids, dtype=np.int32)
    if ref_ids.ndim != 1:
        raise RuntimeError(f"{context}: ref_ids must be 1D")
    if query_ids.size == 0:
        return np.empty((0,), dtype=np.int64)
    pos = np.searchsorted(ref_ids, query_ids)
    valid = pos < ref_ids.shape[0]
    if np.any(valid):
        ok_idx = np.where(valid)[0]
        valid[ok_idx] = ref_ids[pos[ok_idx]] == query_ids[ok_idx]
    if not np.all(valid):
        missing = query_ids[~valid][:10].tolist()
        raise RuntimeError(f"{context}: node_id lookup failed missing={missing}")
    return pos.astype(np.int64, copy=False)


def save_table_auto(df: pd.DataFrame, stem_path: Path) -> Path:
    stem_path.parent.mkdir(parents=True, exist_ok=True)
    if SAVE_PARQUET:
        pq = stem_path.with_suffix(".parquet")
        try:
            df.to_parquet(pq, index=False)
            return pq
        except Exception as exc:
            print(f"  parquet save failed for {pq.name}: {exc.__class__.__name__}: {exc}; fallback to csv")
    csv_path = stem_path.with_suffix(".csv")
    df.to_csv(csv_path, index=False)
    return csv_path

def read_table_auto(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    raise RuntimeError(f"unsupported table suffix for {path}")


def build_true_df_from_events(model_id: int, node_type: int, events, node_ids, start_t: int):
    node_ids = np.asarray(node_ids, dtype=np.int32)
    rows = []
    for ev in events:
        y = np.asarray(ev["y"], dtype=np.float32)
        ts = np.asarray(ev["timestep"], dtype=np.int32)
        n = min(y.shape[0], ts.shape[0])
        if n <= start_t:
            continue
        ts_use = ts[start_t:n]
        y_use = y[start_t:n, :]
        n_rows = int(ts_use.size * node_ids.size)
        if n_rows <= 0:
            continue
        rows.append(
            pd.DataFrame(
                {
                    "model_id": np.full((n_rows,), model_id, dtype=np.int8),
                    "event_id": np.full((n_rows,), int(ev["event_id"]), dtype=np.int32),
                    "node_type": np.full((n_rows,), node_type, dtype=np.int8),
                    "node_id": np.tile(node_ids, ts_use.size),
                    "timestep": np.repeat(ts_use, node_ids.size),
                    "water_level_true": y_use.reshape(-1).astype(np.float32, copy=False),
                }
            )
        )
    if not rows:
        return pd.DataFrame(columns=KEY_COLS + ["water_level_true"])
    out = pd.concat(rows, ignore_index=True)
    out = out.sort_values(KEY_COLS).reset_index(drop=True)
    return out

def build_oof_parts_from_cache(target, node_ids, lambdas_node, n_folds_local):
    cache_entries = sorted(oof_cache_files.get(target, []), key=lambda x: int(x["fold_idx"]))
    if not cache_entries:
        return None
    parts = []
    std_denom = float(STD_DEV[target])
    for cache_entry in cache_entries:
        fold_idx = int(cache_entry["fold_idx"])
        cache_path = Path(cache_entry["path"])
        merged = read_table_auto(cache_path)
        print(
            f"    oof cache target={target} fold={fold_idx + 1}/{n_folds_local}: "
            f"rows={len(merged)} path={cache_path.name}"
        )
        if merged.empty:
            continue
        row_nodes = merged["node_id"].to_numpy(dtype=np.int32, copy=False)
        pos = lookup_sorted_positions(node_ids, row_nodes, f"oof cache target={target} fold={fold_idx + 1}")
        lam_row = lambdas_node[pos].astype(np.float32, copy=False)
        ridge_vec = merged["ridge_pred"].to_numpy(dtype=np.float32, copy=False)
        lgbm_vec = merged["lgbm_pred"].to_numpy(dtype=np.float32, copy=False)
        pred_vec = (1.0 - lam_row) * ridge_vec + lam_row * lgbm_vec
        np.nan_to_num(pred_vec, copy=False, nan=0.0, posinf=SAFE_LEVEL_CLIP, neginf=-SAFE_LEVEL_CLIP)
        np.clip(pred_vec, -SAFE_LEVEL_CLIP, SAFE_LEVEL_CLIP, out=pred_vec)

        part = merged[KEY_COLS + ["water_level_true"]].copy()
        part["water_level_pred"] = pred_vec.astype(np.float32, copy=False)

        run_se = np.zeros((node_ids.shape[0],), dtype=np.float64)
        run_cnt = np.zeros((node_ids.shape[0],), dtype=np.float64)
        n_events_fold = int(part["event_id"].nunique())
        for ev_idx, (_, ge) in enumerate(part.groupby("event_id", sort=False), 1):
            ev_nodes = ge["node_id"].to_numpy(dtype=np.int32, copy=False)
            ev_pos = lookup_sorted_positions(
                node_ids,
                ev_nodes,
                f"oof cache target={target} fold={fold_idx + 1} event_progress",
            )
            ev_diff = (
                ge["water_level_pred"].to_numpy(dtype=np.float64, copy=False)
                - ge["water_level_true"].to_numpy(dtype=np.float64, copy=False)
            )
            ev_err2 = ev_diff * ev_diff
            np.add.at(run_se, ev_pos, ev_err2)
            np.add.at(run_cnt, ev_pos, 1.0)
            if ev_idx % OOF_LOG_EVERY_EVENTS == 0 or ev_idx == n_events_fold:
                valid_node = run_cnt > 0.0
                run_std_rmse = float(
                    np.mean(np.sqrt(run_se[valid_node] / run_cnt[valid_node]) / std_denom)
                )
                print(
                    f"      oof cache progress target={target} fold={fold_idx + 1}: "
                    f"{ev_idx}/{n_events_fold} running_comp_std_rmse={run_std_rmse:.6f}"
                )

        se_node = np.zeros((node_ids.shape[0],), dtype=np.float64)
        cnt_node = np.zeros((node_ids.shape[0],), dtype=np.float64)
        diff_all = (
            part["water_level_pred"].to_numpy(dtype=np.float64, copy=False)
            - part["water_level_true"].to_numpy(dtype=np.float64, copy=False)
        )
        err2_all = diff_all * diff_all
        np.add.at(se_node, pos, err2_all)
        np.add.at(cnt_node, pos, 1.0)
        valid_node = cnt_node > 0.0
        fold_std_rmse = float(
            np.mean(np.sqrt(se_node[valid_node] / cnt_node[valid_node]) / std_denom)
        )
        parts.append(part)
        print(
            f"    oof cache target={target} fold={fold_idx + 1}/{n_folds_local} rows={len(part)} "
            f"fold_comp_std_rmse={fold_std_rmse:.6f}"
        )
        del merged, part
        gc.collect()
    return parts


def build_oof_and_cv_tables():
    oof_parts = []
    if EXPORT_OOF:
        for model_id, node_type in TARGETS:
            target = (model_id, node_type)
            fold_models = lgbm_folds.get(target, [])
            ridge_states = ridge_folds.get(model_id, [])
            model_split = split_by_model.get(model_id)
            if not fold_models:
                print(f"  oof skip target={target}: no lgbm fold models")
                continue
            if not ridge_states:
                print(f"  oof skip target={target}: no ridge folds")
                continue
            if model_split is None:
                print(f"  oof skip target={target}: missing split metadata")
                continue

            node_ids = np.asarray(fold_models[0]["node_ids"], dtype=np.int32)
            lambdas_node = np.asarray(blend_by_target[target]["best_lambda"], dtype=np.float32)
            if lambdas_node.shape[0] != node_ids.shape[0]:
                raise RuntimeError(
                    f"oof lambda size mismatch target={target}: {lambdas_node.shape[0]} vs {node_ids.shape[0]}"
                )

            folds_local = model_split["folds"]
            n_folds_local = min(len(fold_models), len(ridge_states), len(folds_local))
            cached_parts = build_oof_parts_from_cache(target, node_ids, lambdas_node, n_folds_local)
            if cached_parts is not None:
                oof_parts.extend(cached_parts)
                continue
            for fold_idx in range(n_folds_local):
                st = fold_models[fold_idx]
                ridge_state = ridge_states[fold_idx]["state"]
                val_ids = [int(x) for x in folds_local[fold_idx]]
                if not val_ids:
                    print(f"    oof skip target={target} fold={fold_idx + 1}: empty val_ids")
                    continue

                print(
                    f"    oof load target={target} fold={fold_idx + 1}/{n_folds_local}: "
                    f"val_ids={len(val_ids)}"
                )
                val_events_use = load_events_for_ids(
                    st["mdir"],
                    "train",
                    val_ids,
                    node_type,
                    st["node_ids"],
                    st["rain_col"],
                )
                fill_missing(val_events_use, st["means"])

                true_df = build_true_df_from_events(model_id, node_type, val_events_use, st["node_ids"], START_T)
                if true_df.empty:
                    continue

                ridge_val_df = build_model_prediction_df_ridge(
                    ridge_state,
                    split="train",
                    node_type=node_type,
                    selected_node_ids=st["node_ids"],
                    event_ids=val_ids,
                )[KEY_COLS + ["water_level"]]

                lgbm_val_df = predict_target_df_lgbm_weighted(
                    model_id,
                    node_type,
                    val_events_use,
                    st["node_ids"],
                    st["models"],
                    start_t=START_T,
                    y_lags=st["y_lags"],
                    rain_lags=st["rain_lags"],
                    anchor_params=st["anchor_params"],
                    static_mat=st.get("static_mat"),
                )[KEY_COLS + ["water_level"]]

                merged = true_df.merge(
                    ridge_val_df.rename(columns={"water_level": "ridge_pred"}),
                    on=KEY_COLS,
                    how="inner",
                ).merge(
                    lgbm_val_df.rename(columns={"water_level": "lgbm_pred"}),
                    on=KEY_COLS,
                    how="inner",
                )
                merge_ratio = float(len(merged) / max(1, len(true_df)))
                if merged.empty:
                    print(f"    oof fold empty target={target} fold={fold_idx + 1}")
                    continue
                if merge_ratio < OOF_MIN_MERGE_RATIO:
                    raise RuntimeError(
                        f"oof merge coverage too low target={target} fold={fold_idx + 1}: "
                        f"merged={len(merged)} true={len(true_df)} ratio={merge_ratio:.6f} "
                        f"(threshold={OOF_MIN_MERGE_RATIO})"
                    )

                row_nodes = merged["node_id"].to_numpy(dtype=np.int32, copy=False)
                pos = lookup_sorted_positions(node_ids, row_nodes, f"oof target={target} fold={fold_idx + 1}")
                lam_row = lambdas_node[pos].astype(np.float32, copy=False)
                ridge_vec = merged["ridge_pred"].to_numpy(dtype=np.float32, copy=False)
                lgbm_vec = merged["lgbm_pred"].to_numpy(dtype=np.float32, copy=False)
                pred_vec = (1.0 - lam_row) * ridge_vec + lam_row * lgbm_vec
                np.nan_to_num(pred_vec, copy=False, nan=0.0, posinf=SAFE_LEVEL_CLIP, neginf=-SAFE_LEVEL_CLIP)
                np.clip(pred_vec, -SAFE_LEVEL_CLIP, SAFE_LEVEL_CLIP, out=pred_vec)

                part = merged[KEY_COLS + ["water_level_true"]].copy()
                part["water_level_pred"] = pred_vec.astype(np.float32, copy=False)

                std_denom = float(STD_DEV[target])
                run_se = np.zeros((node_ids.shape[0],), dtype=np.float64)
                run_cnt = np.zeros((node_ids.shape[0],), dtype=np.float64)
                n_events_fold = int(part["event_id"].nunique())
                for ev_idx, (_, ge) in enumerate(part.groupby("event_id", sort=False), 1):
                    ev_nodes = ge["node_id"].to_numpy(dtype=np.int32, copy=False)
                    ev_pos = lookup_sorted_positions(
                        node_ids,
                        ev_nodes,
                        f"oof target={target} fold={fold_idx + 1} event_progress",
                    )
                    ev_diff = (
                        ge["water_level_pred"].to_numpy(dtype=np.float64, copy=False)
                        - ge["water_level_true"].to_numpy(dtype=np.float64, copy=False)
                    )
                    ev_err2 = ev_diff * ev_diff
                    np.add.at(run_se, ev_pos, ev_err2)
                    np.add.at(run_cnt, ev_pos, 1.0)
                    if ev_idx % OOF_LOG_EVERY_EVENTS == 0 or ev_idx == n_events_fold:
                        valid_node = run_cnt > 0.0
                        run_std_rmse = float(
                            np.mean(np.sqrt(run_se[valid_node] / run_cnt[valid_node]) / std_denom)
                        )
                        print(
                            f"      oof event progress target={target} fold={fold_idx + 1}: "
                            f"{ev_idx}/{n_events_fold} running_comp_std_rmse={run_std_rmse:.6f}"
                        )

                se_node = np.zeros((node_ids.shape[0],), dtype=np.float64)
                cnt_node = np.zeros((node_ids.shape[0],), dtype=np.float64)
                diff_all = (
                    part["water_level_pred"].to_numpy(dtype=np.float64, copy=False)
                    - part["water_level_true"].to_numpy(dtype=np.float64, copy=False)
                )
                err2_all = diff_all * diff_all
                np.add.at(se_node, pos, err2_all)
                np.add.at(cnt_node, pos, 1.0)
                valid_node = cnt_node > 0.0
                fold_std_rmse = float(
                    np.mean(np.sqrt(se_node[valid_node] / cnt_node[valid_node]) / std_denom)
                )

                oof_parts.append(part)
                print(
                    f"    oof target={target} fold={fold_idx + 1}/{n_folds_local} rows={len(part)} "
                    f"merge_ratio={merge_ratio:.4f} fold_comp_std_rmse={fold_std_rmse:.6f}"
                )

                del val_events_use, true_df, ridge_val_df, lgbm_val_df, merged, part
                gc.collect()

    if oof_parts:
        oof_df_local = pd.concat(oof_parts, ignore_index=True)
        oof_df_local = oof_df_local.drop_duplicates(KEY_COLS, keep="last")
        oof_df_local = oof_df_local.sort_values(KEY_COLS).reset_index(drop=True)
    else:
        oof_df_local = pd.DataFrame(columns=KEY_COLS + ["water_level_true", "water_level_pred"])

    print(f"  oof prediction rows={len(oof_df_local)}")
    if EXPORT_OOF and FAIL_IF_OOF_EMPTY and len(oof_df_local) == 0:
        raise RuntimeError(
            "OOF export produced 0 rows. Check TARGETS / EVENT_SPLIT_SEED / N_FOLDS / split alignment."
        )

    cv_rows = {}
    if len(oof_df_local) > 0:
        for (mid, ntype), g in oof_df_local.groupby(["model_id", "node_type"]):
            std = float(STD_DEV.get((int(mid), int(ntype)), 1.0))
            node_scores = []
            for _, gn in g.groupby("node_id"):
                diff = (
                    gn["water_level_pred"].to_numpy(dtype=np.float64, copy=False)
                    - gn["water_level_true"].to_numpy(dtype=np.float64, copy=False)
                )
                rmse = float(np.sqrt(np.mean(diff * diff)))
                node_scores.append(rmse / std)
            score = float(np.mean(node_scores)) if node_scores else float("nan")
            cv_rows[(int(mid), int(ntype))] = {
                "source_name": BUNDLE_NAME,
                "model_id": int(mid),
                "node_type": int(ntype),
                "num_nodes": int(len(node_scores)),
                "std_rmse": score,
            }
            print(f"  cv target=({int(mid)},{int(ntype)}) std_rmse={score:.6f} nodes={len(node_scores)}")

    for model_id, node_type in TARGETS:
        key = (int(model_id), int(node_type))
        if key in cv_rows:
            continue
        info = blend_by_target.get(key)
        if info is None:
            continue
        fallback_score = float(info.get("selected_mean", float("nan")))
        cv_rows[key] = {
            "source_name": BUNDLE_NAME,
            "model_id": int(model_id),
            "node_type": int(node_type),
            "num_nodes": int(len(np.asarray(info.get("best_lambda", []), dtype=np.float32))),
            "std_rmse": fallback_score,
        }
        print(
            f"  cv target=({int(model_id)},{int(node_type)}) fallback_std_rmse={fallback_score:.6f} "
            f"nodes={cv_rows[key]['num_nodes']}"
        )

    cv_summary_local = pd.DataFrame(
        list(cv_rows.values()),
        columns=["source_name", "model_id", "node_type", "num_nodes", "std_rmse"],
    )
    if len(cv_summary_local) > 0:
        print(f"  cv overall std_rmse={float(cv_summary_local['std_rmse'].mean()):.6f}")
    return oof_df_local, cv_summary_local


total_start = time.time()
print(f"DATASET_ROOT: {DATASET_ROOT}")
print(f"TARGETS: {TARGETS}")
print(f"FORCE_METHOD: {FORCE_METHOD or 'auto'}")
print(f"LAMBDA_SOLVER: closed_form (LAMBDA_CLIP_EPS={LAMBDA_CLIP_EPS})")
print(f"LGBM_DEVICE: {LGBM_DEVICE} (source={LGBM_DEVICE_SOURCE})")
print(f"KFOLD: N_FOLDS={N_FOLDS} EVENT_SPLIT_SEED={EVENT_SPLIT_SEED}")
print(
    "RIDGE: "
    f"RIDGE_ALPHA={RIDGE_ALPHA} NA_MODEL1={NA_MODEL1} NA_MODEL2={NA_MODEL2} "
    f"NB_MODEL1={NB_MODEL1} NB_MODEL2={NB_MODEL2}"
)
print(
    "LGBM LAGS: "
    f"START_T={START_T} Y_LAGS_1D={Y_LAGS_1D} Y_LAGS_2D={Y_LAGS_2D} "
    f"RAIN_LAGS_MODEL1={RAIN_LAGS_MODEL1} RAIN_LAGS_MODEL2={RAIN_LAGS_MODEL2}"
)
print(
    "LGBM ROW_LIMITS: "
    f"LGBM_MAX_TRAIN_ROWS_1D={LGBM_MAX_TRAIN_ROWS_1D} "
    f"LGBM_MAX_TRAIN_ROWS_2D={LGBM_MAX_TRAIN_ROWS_2D}"
)
print("RIDGE ROW_LIMITS: full train rows (no row cap)")
print(
    "LGBM PARAMS: "
    f"N_ESTIMATORS_1D={N_ESTIMATORS_1D} N_ESTIMATORS_2D={N_ESTIMATORS_2D} "
    f"LEARNING_RATE={LEARNING_RATE} NUM_LEAVES={NUM_LEAVES} MAX_DEPTH={MAX_DEPTH} "
    f"MIN_CHILD_SAMPLES_1D={MIN_CHILD_SAMPLES_1D} MIN_CHILD_SAMPLES_2D={MIN_CHILD_SAMPLES_2D} "
    f"MIN_DATA_IN_BIN={MIN_DATA_IN_BIN} SUBSAMPLE={SUBSAMPLE} COLSAMPLE_BYTREE={COLSAMPLE_BYTREE} "
    f"REG_ALPHA={REG_ALPHA} REG_LAMBDA={REG_LAMBDA} LGBM_N_JOBS={LGBM_N_JOBS}"
)
print(
    "GPU_OPTS: "
    f"MAX_BIN={MAX_BIN} MAX_CAT_THRESHOLD={MAX_CAT_THRESHOLD} "
    f"GPU_USE_DP={GPU_USE_DP} NODE_POS_AS_CATEGORY={NODE_POS_AS_CATEGORY}"
)
print(
    "RAIN_ANCHOR: "
    f"QLOW={RAIN_ANCHOR_QLOW} QMID={RAIN_ANCHOR_QMID} QHIGH={RAIN_ANCHOR_QHIGH} "
    f"BW_SCALE={RAIN_ANCHOR_BW_SCALE} MIN_BW={RAIN_ANCHOR_MIN_BW}"
)
print(
    "STATIC: "
    f"USE_STATIC_FEATURES_RIDGE={USE_STATIC_FEATURES_RIDGE} "
    f"USE_STATIC_FEATURES_LGBM={USE_STATIC_FEATURES_LGBM} "
    f"STATIC_FEATURE_MODE={STATIC_FEATURE_MODE} STATIC_STANDARDIZE={STATIC_STANDARDIZE}"
)
print(f"ALLOW_PARTIAL_TARGETS={ALLOW_PARTIAL_TARGETS}")

models_root = resolve_models_root(DATASET_ROOT)
sample_path = resolve_sample_submission(DATASET_ROOT, models_root, SAMPLE_SUBMISSION_PATH)
print(f"SAMPLE_SUBMISSION: {sample_path}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")
print(f"SAVE_BUNDLE: {SAVE_BUNDLE} BUNDLE_DIR={BUNDLE_DIR}")

needed_models = sorted({m for m, _ in TARGETS})

print("Step 1: Build K-fold splits per model")
split_by_model = {}
for model_id in needed_models:
    mdir = model_dir(models_root, model_id)
    train_ids = list_event_ids(mdir, "train")
    test_ids = list_event_ids(mdir, "test")
    if len(train_ids) < 2:
        raise RuntimeError(f"not enough train events for model {model_id}")
    folds = split_events_kfold(train_ids, N_FOLDS, EVENT_SPLIT_SEED + model_id * 1000)
    split_by_model[model_id] = {
        "mdir": mdir,
        "train_all": train_ids,
        "folds": folds,
        "test": test_ids,
    }
    fold_sizes = ",".join(str(len(f)) for f in folds)
    print(
        f"  model {model_id}: train_events={len(train_ids)} test_events={len(test_ids)} "
        f"folds={len(folds)} sizes=[{fold_sizes}]"
    )


print("Step 2: Fit joint ridge per model and fold")
ridge_folds = {}
for model_id in needed_models:
    mdir = split_by_model[model_id]["mdir"]
    folds = split_by_model[model_id]["folds"]
    na, nb, alpha = model_ridge_params(model_id)

    node_ids_1 = load_node_ids(mdir, 1)
    node_ids_2 = load_node_ids(mdir, 2)
    static_mode_ridge = STATIC_FEATURE_MODE if USE_STATIC_FEATURES_RIDGE else "off"
    static1, static_cols_1 = load_static_features(mdir, model_id, 1, node_ids_1, STATIC_COLS_1D_REQ)
    static2, static_cols_2 = load_static_features(mdir, model_id, 2, node_ids_2, STATIC_COLS_2D_REQ)
    n_total = len(node_ids_1) + len(node_ids_2)
    feat_dim = na * n_total + (nb + 1) + static_aug_dim(static1, static2, na, static_mode_ridge) + 1
    est_xtx_gib = (feat_dim * feat_dim * 8) / (1024 ** 3)
    print(
        f"  model {model_id}: na={na} nb={nb} alpha={alpha} "
        f"nodes(1d={len(node_ids_1)},2d={len(node_ids_2)}) feat_dim={feat_dim} est_xtx={est_xtx_gib:.2f} GiB "
        f"static_mode={static_mode_ridge} static_cols_1d={len(static_cols_1)} static_cols_2d={len(static_cols_2)}"
    )

    fold_states = []
    for fold_idx, val_ev_ids in enumerate(folds):
        tr_events = fold_train_events(folds, fold_idx)
        print(
            f"    ridge fold {fold_idx + 1}/{len(folds)}: "
            f"train_events={len(tr_events)} val_events={len(val_ev_ids)}"
        )
        node_means_1 = compute_node_means_from_csv(mdir, 1, node_ids_1, tr_events)
        node_means_2 = compute_node_means_from_csv(mdir, 2, node_ids_2, tr_events)

        t0 = time.time()
        state = fit_joint_ridge(
            mdir,
            model_id,
            node_ids_1,
            node_ids_2,
            node_means_1,
            node_means_2,
            static1,
            static2,
            tr_events,
            na,
            nb,
            alpha,
            static_mode_ridge,
        )
        fold_states.append(
            {
                "state": state,
                "val_events": list(val_ev_ids),
            }
        )
        print(
            f"      ridge fold fit done in {time.time() - t0:.1f}s "
            f"rows={state['train_rows']} used_events={state['used_events']}"
        )
        ridge_s1, ridge_s2, _, _ = evaluate_model_ridge_event_average(state, val_ev_ids, fold_idx=fold_idx, n_folds=len(folds))
        print(
            f"      ridge fold val model {model_id}: "
            f"1d_std_rmse={format_score(ridge_s1)} 2d_std_rmse={format_score(ridge_s2)}"
        )
    ridge_folds[model_id] = fold_states


print("Step 3: Fit rain-weighted LightGBM (3 anchors) per target and fold, accumulate OOF lambda scores")
lgbm_folds = {}
lambda_acc = {}
oof_cache_files = {}
for model_id, node_type in TARGETS:
    mdir = split_by_model[model_id]["mdir"]
    folds = split_by_model[model_id]["folds"]
    test_ev_ids = split_by_model[model_id]["test"]
    all_train_ids = split_by_model[model_id]["train_all"]

    rain_col = detect_rain_col(mdir, "train", all_train_ids[0])
    node_ids = load_node_ids(mdir, node_type)
    if node_type == 1:
        ridge_ids = ridge_folds[model_id][0]["state"]["node_ids_1"]
        y_lags = Y_LAGS_1D
        max_rows = LGBM_MAX_TRAIN_ROWS_1D
    else:
        ridge_ids = ridge_folds[model_id][0]["state"]["node_ids_2"]
        y_lags = Y_LAGS_2D
        max_rows = LGBM_MAX_TRAIN_ROWS_2D
    if not np.array_equal(ridge_ids, node_ids):
        raise RuntimeError(f"node_id ordering mismatch between ridge and lgbm for target={(model_id, node_type)}")

    rain_lags = RAIN_LAGS_MODEL1 if model_id == 1 else RAIN_LAGS_MODEL2
    static_mat = None
    static_cols = []
    if USE_STATIC_FEATURES_LGBM:
        static_req = STATIC_COLS_1D_REQ if node_type == 1 else STATIC_COLS_2D_REQ
        static_mat, static_cols = load_static_features(mdir, model_id, node_type, node_ids, static_req)

    print(
        f"  model {model_id} node_type {node_type}: "
        f"nodes={len(node_ids)} folds={len(folds)} "
        f"y_lags={y_lags} rain_lags={rain_lags} max_rows={max_rows} static_dim={0 if static_mat is None else static_mat.shape[1]}"
    )
    fold_states = []
    target_oof_cache = []
    sse0_total = np.zeros((len(node_ids),), dtype=np.float64)
    sse1_total = np.zeros((len(node_ids),), dtype=np.float64)
    sse2_total = np.zeros((len(node_ids),), dtype=np.float64)
    cnt_total = np.zeros((len(node_ids),), dtype=np.float64)
    used_events_total = 0

    for fold_idx, val_ev_ids in enumerate(folds):
        tr_ev_ids = fold_train_events(folds, fold_idx)
        print(
            f"    lgbm fold {fold_idx + 1}/{len(folds)}: "
            f"train_events={len(tr_ev_ids)} val_events={len(val_ev_ids)}"
        )

        train_events = load_events_for_ids(mdir, "train", tr_ev_ids, node_type, node_ids, rain_col)
        val_events = load_events_for_ids(mdir, "train", val_ev_ids, node_type, node_ids, rain_col)

        means = compute_node_means(train_events, len(node_ids))
        fill_missing(train_events, means)
        fill_missing(val_events, means)

        anchor_params = build_rain_anchor_params(train_events)
        print(
            "      rain anchors: "
            f"centers=[{anchor_params['centers'][0]:.4f}, {anchor_params['centers'][1]:.4f}, {anchor_params['centers'][2]:.4f}] "
            f"bandwidth={anchor_params['bandwidth']:.4f}"
        )

        x_train, y_train, rows_possible, rows_kept, row_event_ids = build_samples_global(
            train_events,
            start_t=START_T,
            y_lags=y_lags,
            rain_lags=rain_lags,
            max_rows=max_rows,
            seed=EVENT_SPLIT_SEED + model_id * 1000 + node_type * 100 + fold_idx,
            return_event_ids=True,
            static_mat=static_mat,
        )
        print(
            f"      lgbm train matrix: x={x_train.shape} y={y_train.shape} "
            f"rows_kept={rows_kept}/{rows_possible}"
        )
        sample_w = build_anchor_sample_weights(
            row_event_ids=row_event_ids,
            score_by_event=anchor_params["score_by_event"],
            centers=anchor_params["centers"],
            bandwidth=anchor_params["bandwidth"],
        )
        anchor_mean_w = np.mean(sample_w, axis=0)
        print(
            "      anchor sample_weight mean: "
            f"dry={float(anchor_mean_w[0]):.4f} mid={float(anchor_mean_w[1]):.4f} wet={float(anchor_mean_w[2]):.4f}"
        )

        models = []
        t0 = time.time()
        base_seed = EVENT_SPLIT_SEED + model_id * 1000 + node_type * 100 + fold_idx
        for anchor_idx, anchor_name in enumerate(("dry", "mid", "wet")):
            w_anchor = sample_w[:, anchor_idx].astype(np.float32, copy=False)
            if float(np.sum(w_anchor)) <= 0.0:
                w_anchor = None
                print(f"      anchor={anchor_name}: fallback to unweighted fit")
            model_anchor = fit_global_lgbm(
                x_train,
                y_train,
                seed=base_seed + anchor_idx,
                node_type=node_type,
                device=LGBM_DEVICE,
                sample_weight=w_anchor,
            )
            models.append(model_anchor)
        print(f"      lgbm 3-anchor fold fit done in {time.time() - t0:.1f}s")
        lgbm_fold_score, _ = evaluate_target_lgbm_per_node_weighted(
            model_id=model_id,
            node_type=node_type,
            val_events=val_events,
            models=models,
            start_t=START_T,
            y_lags=y_lags,
            rain_lags=rain_lags,
            anchor_params=anchor_params,
            static_mat=static_mat,
        )
        print(
            f"      lgbm fold val target=({model_id},{node_type}): "
            f"std_rmse={format_score(lgbm_fold_score)}"
        )

        eval_st = {
            "models": models,
            "node_ids": node_ids,
            "val_events": val_events,
            "y_lags": y_lags,
            "rain_lags": rain_lags,
            "anchor_params": anchor_params,
            "static_mat": static_mat,
        }
        ridge_state = ridge_folds[model_id][fold_idx]["state"]
        sse0_fold, sse1_fold, sse2_fold, cnt_fold, used_events, oof_fold_raw = compute_lambda_sse_per_node(
            target=(model_id, node_type),
            ridge_state=ridge_state,
            lgbm_st=eval_st,
            val_event_ids=val_ev_ids,
            return_oof_part=bool(SAVE_BUNDLE and EXPORT_OOF),
        )
        if SAVE_BUNDLE and EXPORT_OOF and oof_fold_raw is not None:
            cache_path = save_table_auto(
                oof_fold_raw,
                OOF_CACHE_DIR / f"oof_raw__m{model_id}_n{node_type}_f{fold_idx + 1:02d}",
            )
            target_oof_cache.append(
                {
                    "fold_idx": int(fold_idx),
                    "path": str(cache_path),
                    "rows": int(len(oof_fold_raw)),
                }
            )
            del oof_fold_raw
        sse0_total += sse0_fold
        sse1_total += sse1_fold
        sse2_total += sse2_fold
        cnt_total += cnt_fold
        used_events_total += used_events

        fold_info = summarize_lambda_from_sse(
            target=(model_id, node_type),
            sse0=sse0_fold,
            sse1=sse1_fold,
            sse2=sse2_fold,
            cnt=cnt_fold,
            used_events=used_events,
        )
        print(
            f"      blend fold val target=({model_id},{node_type}) used_events={used_events}: "
            f"ridge_std_rmse={format_score(fold_info['ridge_mean'])} "
            f"lgbm_std_rmse={format_score(fold_info['lgbm_mean'])} "
            f"blend_std_rmse={format_score(fold_info['selected_mean'])}"
        )

        fold_states.append(
            {
                "models": models,
                "node_ids": node_ids,
                "y_lags": y_lags,
                "rain_lags": rain_lags,
                "means": means,
                "rain_col": rain_col,
                "mdir": mdir,
                "test_event_ids": test_ev_ids,
                "anchor_params": anchor_params,
                "static_mat": static_mat,
                "static_cols": list(static_cols),
            }
        )

        del train_events, val_events, x_train, y_train, row_event_ids, sample_w, eval_st, sse0_fold, sse1_fold, sse2_fold, cnt_fold
        gc.collect()

    lgbm_folds[(model_id, node_type)] = fold_states
    lambda_acc[(model_id, node_type)] = {
        "sse0": sse0_total,
        "sse1": sse1_total,
        "sse2": sse2_total,
        "cnt": cnt_total,
        "used_events": used_events_total,
    }
    oof_cache_files[(model_id, node_type)] = target_oof_cache


print("Step 4: Select per-node lambda from K-fold OOF average")

blend_by_target = {}
ridge_scores_used = []
lgbm_scores_used = []
selected_scores = []

for model_id, node_type in TARGETS:
    target = (model_id, node_type)
    acc = lambda_acc[target]
    info = summarize_lambda_from_sse(
        target=target,
        sse0=acc["sse0"],
        sse1=acc["sse1"],
        sse2=acc["sse2"],
        cnt=acc["cnt"],
        used_events=acc["used_events"],
    )
    blend_by_target[target] = info

    rs = info["ridge_mean"]
    ls = info["lgbm_mean"]
    ss = info["selected_mean"]
    ridge_scores_used.append(rs)
    lgbm_scores_used.append(ls)
    selected_scores.append(ss)

    lam = info["best_lambda"]
    print(f"  Ridge model {model_id} node_type {node_type}: {format_score(rs)}")
    print(f"  LGBM  model {model_id} node_type {node_type}: {format_score(ls)}")
    print(
        f"  Blended model {model_id} node_type {node_type}: {format_score(ss)} "
        f"(lambda min={float(np.min(lam)):.3f} mean={float(np.mean(lam)):.3f} max={float(np.max(lam)):.3f})"
    )

if ridge_scores_used:
    rs_arr = np.asarray(ridge_scores_used, dtype=np.float64)
    if np.any(np.isfinite(rs_arr)):
        print(f"  Mean ridge val score over targets: {float(np.mean(rs_arr[np.isfinite(rs_arr)])):.6f}")
if lgbm_scores_used:
    ls_arr = np.asarray(lgbm_scores_used, dtype=np.float64)
    if np.any(np.isfinite(ls_arr)):
        print(f"  Mean lgbm  val score over targets: {float(np.mean(ls_arr[np.isfinite(ls_arr)])):.6f}")
if selected_scores:
    ss_arr = np.asarray(selected_scores, dtype=np.float64)
    if np.any(np.isfinite(ss_arr)):
        print(f"  Mean blended val score over targets: {float(np.mean(ss_arr[np.isfinite(ss_arr)])):.6f}")


print("Step 5: Build submission (fold-avg ridge/lgbm + per-node lambda blend)")
sample_cols_probe = pd.read_csv(sample_path, nrows=0).columns.tolist()
sample_dtype = {
    "row_id": np.int64,
    "model_id": np.int8,
    "event_id": np.int32,
    "node_type": np.int8,
    "node_id": np.int32,
    "timestep": np.int32,
    "water_level": np.float32,
}
sample_dtype = {k: v for k, v in sample_dtype.items() if k in sample_cols_probe}
sample_work = pd.read_csv(sample_path, dtype=sample_dtype)
sample_cols = sample_work.columns.tolist()
sample_n_rows = len(sample_work)
sample_work["__orig_idx"] = np.arange(len(sample_work), dtype=np.int64)
if "water_level" in sample_cols:
    sample_work["water_level"] = pd.to_numeric(sample_work["water_level"], errors="coerce").astype(np.float32)
    if ALLOW_PARTIAL_TARGETS:
        sample_work["__baseline_wl"] = sample_work["water_level"].to_numpy(dtype=np.float32, copy=False)
else:
    sample_work["water_level"] = np.nan

has_key_cols = set(KEY_COLS).issubset(sample_cols)
has_base_keys = set(["model_id", "event_id", "node_type", "node_id"]).issubset(sample_cols)
if not has_key_cols and not has_base_keys:
    raise RuntimeError("sample submission missing required key columns")

if has_base_keys and not has_key_cols:
    base_keys = ["model_id", "event_id", "node_type", "node_id"]
    sample_work = sample_work.sort_values(base_keys + (["row_id"] if "row_id" in sample_work.columns else []))
    sample_work["__step_idx"] = sample_work.groupby(base_keys).cumcount()

for model_id, node_type in TARGETS:
    target = (model_id, node_type)
    fold_models = lgbm_folds[target]
    if not fold_models:
        raise RuntimeError(f"no fold models for target={target}")
    node_ids = fold_models[0]["node_ids"]
    lambdas_node = blend_by_target[target]["best_lambda"]
    if len(lambdas_node) != len(node_ids):
        raise RuntimeError(f"lambda size mismatch for target={target}: {len(lambdas_node)} vs {len(node_ids)}")
    if len(ridge_folds[model_id]) != len(fold_models):
        raise RuntimeError(
            f"fold count mismatch for model {model_id}: ridge={len(ridge_folds[model_id])} lgbm={len(fold_models)}"
        )

    target_mask = (sample_work["model_id"] == model_id) & (sample_work["node_type"] == node_type)
    expected_rows = int(target_mask.sum())
    if expected_rows <= 0:
        raise RuntimeError(f"no submission rows for target={target}")

    if has_key_cols:
        base_target = sample_work.loc[target_mask, KEY_COLS]
    else:
        base_keys = ["model_id", "event_id", "node_type", "node_id"]
        base_target = sample_work.loc[target_mask, base_keys + ["__step_idx"]]

    ridge_sum = np.zeros((expected_rows,), dtype=np.float32)
    lgbm_sum = np.zeros((expected_rows,), dtype=np.float32)
    n_fold_models = len(fold_models)
    for fold_idx in range(n_fold_models):
        print(
            f"  Predict target model {model_id} node_type {node_type} "
            f"fold {fold_idx + 1}/{n_fold_models}"
        )

        ridge_state = ridge_folds[model_id][fold_idx]["state"]
        ridge_df = build_model_prediction_df_ridge(
            ridge_state,
            split="test",
            node_type=node_type,
        )[KEY_COLS + ["water_level"]]
        ridge_vec = align_prediction_vector(base_target, ridge_df, has_key_cols)
        ridge_sum += ridge_vec

        st = fold_models[fold_idx]
        test_events = load_events_for_ids(
            st["mdir"],
            "test",
            st["test_event_ids"],
            node_type,
            st["node_ids"],
            st["rain_col"],
        )
        fill_missing(test_events, st["means"])
        lgbm_df = predict_target_df_lgbm_weighted(
            model_id,
            node_type,
            test_events,
            st["node_ids"],
            st["models"],
            start_t=START_T,
            y_lags=st["y_lags"],
            rain_lags=st["rain_lags"],
            anchor_params=st["anchor_params"],
            static_mat=st.get("static_mat"),
        )[KEY_COLS + ["water_level"]]
        lgbm_vec = align_prediction_vector(base_target, lgbm_df, has_key_cols)
        lgbm_sum += lgbm_vec

        del ridge_df, lgbm_df, ridge_vec, lgbm_vec, test_events
        gc.collect()

    ridge_mean = ridge_sum / float(n_fold_models)
    lgbm_mean = lgbm_sum / float(n_fold_models)

    row_nodes = base_target["node_id"].to_numpy(dtype=np.int32, copy=False)
    pos = lookup_sorted_positions(node_ids, row_nodes, f"submission target={target}")
    lam_row = lambdas_node[pos].astype(np.float32, copy=False)

    blended_wl = (1.0 - lam_row) * ridge_mean + lam_row * lgbm_mean
    np.nan_to_num(blended_wl, copy=False, nan=0.0, posinf=SAFE_LEVEL_CLIP, neginf=-SAFE_LEVEL_CLIP)
    np.clip(blended_wl, -SAFE_LEVEL_CLIP, SAFE_LEVEL_CLIP, out=blended_wl)

    sample_work.loc[target_mask, "water_level"] = blended_wl
    print(
        f"  Blended model {model_id} node_type {node_type}: rows={expected_rows} folds={n_fold_models} "
        f"lambda min={float(np.min(lambdas_node)):.3f} mean={float(np.mean(lambdas_node)):.3f} max={float(np.max(lambdas_node)):.3f}"
    )
    del base_target, ridge_sum, lgbm_sum, ridge_mean, lgbm_mean, lam_row, blended_wl
    gc.collect()

sub = sample_work

missing = int(sub["water_level"].isna().sum())
if missing > 0:
    if ALLOW_PARTIAL_TARGETS and "__baseline_wl" in sub.columns:
        mask = sub["water_level"].isna()
        baseline = pd.to_numeric(sub.loc[mask, "__baseline_wl"], errors="coerce").to_numpy(dtype=np.float32, copy=False)
        sub.loc[mask, "water_level"] = baseline
        missing = int(sub["water_level"].isna().sum())
    if missing > 0 and ALLOW_PARTIAL_TARGETS and not STRICT_PARTIAL_SUBMISSION:
        mask = sub["water_level"].isna()
        fill_value = np.float32(PARTIAL_SUBMISSION_FILL_VALUE)
        sub.loc[mask, "water_level"] = fill_value
        print(
            f"warning: partial targets without baseline water_level in sample; "
            f"filled {missing} untouched submission rows with {float(fill_value):.6f}"
        )
        missing = int(sub["water_level"].isna().sum())
    if missing > 0:
        raise RuntimeError(
            f"submission has missing water_level rows: {missing}. "
            "If running partial targets, set ALLOW_PARTIAL_TARGETS=1 and provide baseline water_level in sample, "
            "or set STRICT_PARTIAL_SUBMISSION=0 to allow fill."
        )

if len(sub) != sample_n_rows:
    raise RuntimeError("row count mismatch after merge")

sub = sub.sort_values("__orig_idx").drop(columns=["__orig_idx"], errors="ignore")
sub = sub.drop(columns=["__step_idx"], errors="ignore")
sub = sub.drop(columns=["__baseline_wl"], errors="ignore")

if "water_level" not in sample_cols:
    sample_cols.append("water_level")
missing_cols = [c for c in sample_cols if c not in sub.columns]
if missing_cols:
    raise RuntimeError(f"submission missing columns required by sample: {missing_cols}")
sub = sub[sample_cols]

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(OUTPUT_PATH, index=False)
print(f"submission written: {OUTPUT_PATH} rows={len(sub)}")

if SAVE_BUNDLE:
    print("Step 6: Export bundle tables")
    oof_df, cv_summary = build_oof_and_cv_tables()

    req_test_keys = ["model_id", "event_id", "node_type", "node_id"]
    if not set(req_test_keys).issubset(sub.columns):
        raise RuntimeError(f"submission is missing required test keys: {req_test_keys}")

    pred_mask = np.zeros((len(sub),), dtype=bool)
    for model_id, node_type in TARGETS:
        pred_mask |= ((sub["model_id"] == int(model_id)) & (sub["node_type"] == int(node_type)))

    test_pred_df = sub.loc[pred_mask].copy()
    if "water_level" not in test_pred_df.columns:
        raise RuntimeError("submission frame is missing water_level")
    test_pred_df = test_pred_df.rename(columns={"water_level": "water_level_pred"})

    keep_test_cols = [
        c
        for c in ["row_id", "model_id", "event_id", "node_type", "node_id", "timestep", "water_level_pred"]
        if c in test_pred_df.columns
    ]
    keep_test_cols = list(dict.fromkeys(keep_test_cols))
    if "water_level_pred" not in keep_test_cols:
        keep_test_cols.append("water_level_pred")
    test_pred_df = test_pred_df[keep_test_cols]
    if "row_id" in test_pred_df.columns:
        test_pred_df = test_pred_df.sort_values("row_id").reset_index(drop=True)
    else:
        sort_cols = [c for c in ["model_id", "event_id", "node_type", "node_id", "timestep"] if c in test_pred_df.columns]
        test_pred_df = test_pred_df.sort_values(sort_cols).reset_index(drop=True)
    print(f"  test prediction rows={len(test_pred_df)}")

    BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
    oof_path = save_table_auto(oof_df, BUNDLE_DIR / "oof_predictions")
    test_path = save_table_auto(test_pred_df, BUNDLE_DIR / "test_predictions")
    sub_path = BUNDLE_DIR / "submission.csv"
    sub.to_csv(sub_path, index=False)
    cv_path = BUNDLE_DIR / "cv_summary.csv"
    cv_summary.to_csv(cv_path, index=False)

    manifest = {
        "schema_version": "ufb_ensemble_v1",
        "bundle_name": BUNDLE_NAME,
        "created_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "n_folds": int(N_FOLDS),
        "event_split_seed": int(EVENT_SPLIT_SEED),
        "targets": [f"{int(model_id)}:{int(node_type)}" for model_id, node_type in TARGETS],
        "use_static_features_ridge": bool(USE_STATIC_FEATURES_RIDGE),
        "use_static_features_lgbm": bool(USE_STATIC_FEATURES_LGBM),
        "static_feature_mode": STATIC_FEATURE_MODE,
        "static_standardize": bool(STATIC_STANDARDIZE),
        "static_cols_1d": list(STATIC_COLS_1D_REQ),
        "static_cols_2d": list(STATIC_COLS_2D_REQ),
        "oof_predictions_path": oof_path.name,
        "test_predictions_path": test_path.name,
        "submission_path": sub_path.name,
        "cv_summary_path": cv_path.name,
        "notes": "trained and bundled in a single notebook run",
    }
    manifest_path = BUNDLE_DIR / "bundle_manifest.json"
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))

    print(f"saved bundle dir: {BUNDLE_DIR}")
    print(f"  - {manifest_path.name}")
    print(f"  - {oof_path.name}")
    print(f"  - {test_path.name}")
    print(f"  - {sub_path.name}")
    print(f"  - {cv_path.name}")
else:
    print("SAVE_BUNDLE=0: skipped bundle export")

print(f"All done in {time.time() - total_start:.1f}s")


DATASET_ROOT: /kaggle/input/datasets/mtmrs1/urban-flood-bench-re
TARGETS: [(1, 1), (1, 2)]
FORCE_METHOD: auto
LAMBDA_SOLVER: closed_form (LAMBDA_CLIP_EPS=1e-12)
LGBM_DEVICE: cpu (source=auto)
KFOLD: N_FOLDS=5 EVENT_SPLIT_SEED=2026
RIDGE: RIDGE_ALPHA=0.1 NA_MODEL1=3 NA_MODEL2=3 NB_MODEL1=100 NB_MODEL2=100
LGBM LAGS: START_T=10 Y_LAGS_1D=10 Y_LAGS_2D=4 RAIN_LAGS_MODEL1=60 RAIN_LAGS_MODEL2=80
LGBM ROW_LIMITS: LGBM_MAX_TRAIN_ROWS_1D=4000000 LGBM_MAX_TRAIN_ROWS_2D=8000000
RIDGE ROW_LIMITS: full train rows (no row cap)
LGBM PARAMS: N_ESTIMATORS_1D=800 N_ESTIMATORS_2D=500 LEARNING_RATE=0.03 NUM_LEAVES=63 MAX_DEPTH=-1 MIN_CHILD_SAMPLES_1D=50 MIN_CHILD_SAMPLES_2D=100 MIN_DATA_IN_BIN=15 SUBSAMPLE=0.9 COLSAMPLE_BYTREE=0.9 REG_ALPHA=0.0 REG_LAMBDA=0.0 LGBM_N_JOBS=0
GPU_OPTS: MAX_BIN=255 MAX_CAT_THRESHOLD=64 GPU_USE_DP=True NODE_POS_AS_CATEGORY=True
RAIN_ANCHOR: QLOW=0.1 QMID=0.5 QHIGH=0.9 BW_SCALE=0.75 MIN_BW=0.0001
STATIC: USE_STATIC_FEATURES_RIDGE=True USE_STATIC_FEATURES_LGBM=True STATIC_FEATUR